# Window C seat placement — re-run notebook (A3 → A4)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Starive/A3-for-ARCH7476/blob/main/a3_seat_C.ipynb)

**How to run:** Runtime → Run all. Nothing needs to be uploaded or installed.

**Expected end of section 3:** `RESULT: ALL CHECKS PASS · REPRODUCED`. Section 4 shows the key figure.

**Decision:** where to place one 2.0 × 1.0 m seat in Window C so that it supports short stays while keeping the assumed walking corridor clear.

**What it does:** the A3 baseline (36 scenarios), the A4 window-size test (504 scenarios for 4.0–7.0 m windows, plus the offset rule), 39 checks, a comparison with the reference tables, and the figures. This notebook runs exactly the same file (`run_all.py`) and the same checks as a local run; no step is skipped.

**Limits:** schematic geometry in local metres; the movement edge, w, c₁, c₂, s and the window size are assumptions. The results screen geometry; they are not evidence of pedestrian flow, short stays or comfort.

## 1. Environment

In [ ]:
import importlib.util, subprocess, sys, platform
missing = [m for m in ['numpy', 'matplotlib'] if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + missing, check=True)
import numpy, matplotlib
print('python', platform.python_version(), '| numpy', numpy.__version__, '| matplotlib', matplotlib.__version__)

## 2. Code and inputs
If the notebook sits inside the repository folder, those files are used. Otherwise the copies embedded below are written to `A4_rerun/`.

In [ ]:
from pathlib import Path
FILES = {"a3src/__init__.py": "", "a3src/a3_workflow.py": "\"\"\"A3 Window C workflow core.\n\nLocal-metre schematic frame: x = distance inward from the east edge proxy (0..window),\ny = distance south from the north edge proxy (0..window). No map coordinates are loaded.\nOne 2 x 1 m seat is tested at six alternative placements. Each seat keeps the same position\nin space under both edge proxies; only the edge used as the movement proxy changes.\n\nThe core uses exact interval arithmetic (Python standard library). `grid_cross_check` re-measures\nall 36 scenarios by counting 1 cm grid cells with numpy, as an independent check. No GIS library is\nneeded, so the same checks run on a local machine and in Colab.\n\"\"\"\nimport csv\nimport json\nfrom pathlib import Path\n\nEPS = 1e-9\n\n\ndef load_inputs(root):\n    root = Path(root)\n    cfg = json.loads((root / \"inputs\" / \"config.json\").read_text(encoding=\"utf8\"))\n    with open(root / \"inputs\" / \"variants.csv\", encoding=\"utf8\", newline=\"\") as f:\n        variants = [dict(r, centre_from_east_m=float(r[\"centre_from_east_m\"])) for r in csv.DictReader(f)]\n    return cfg, variants\n\n\ndef seat_rect(variant, cfg, centre_from_east=None):\n    \"\"\"Seat rectangle (x0, x1, y0, y1). 'parallel' = long side N-S, along the east edge.\"\"\"\n    xc = variant[\"centre_from_east_m\"] if centre_from_east is None else centre_from_east\n    yc = cfg[\"seat_centre_from_north_m\"]\n    if variant[\"orientation\"] == \"parallel\":\n        hx, hy = cfg[\"seat_short_m\"] / 2, cfg[\"seat_long_m\"] / 2\n    else:\n        hx, hy = cfg[\"seat_long_m\"] / 2, cfg[\"seat_short_m\"] / 2\n    return (xc - hx, xc + hx, yc - hy, yc + hy)\n\n\ndef _overlap(a0, a1, b0, b1):\n    return max(0.0, min(a1, b1) - max(a0, b0))\n\n\ndef evaluate(rect, edge, w, cfg, c1=None, c2=None):\n    \"\"\"Corridor overlap (m2) and proposed band share (%) of one seat for one edge proxy and width.\"\"\"\n    c1 = cfg[\"c1_m\"] if c1 is None else c1\n    c2 = cfg[\"c2_m\"] if c2 is None else c2\n    x0, x1, y0, y1 = rect\n    d0, d1, along = (x0, x1, y1 - y0) if edge == \"east\" else (y0, y1, x1 - x0)\n    W = cfg[\"window_m\"]\n    area = (x1 - x0) * (y1 - y0)\n    overlap = _overlap(d0, d1, 0.0, w) * along\n    band = _overlap(d0, d1, min(w + c1, W), min(w + c2, W)) * along\n    return overlap, 100.0 * band / area\n\n\ndef passes(overlap, share, cfg, s=None):\n    s = cfg[\"s_pct\"] if s is None else s\n    return overlap <= EPS and share >= s - EPS\n\n\ndef run_scenarios(cfg, variants):\n    rows = []\n    for edge in cfg[\"edge_proxies\"]:\n        for w in cfg[\"widths_m\"]:\n            for v in variants:\n                rect = seat_rect(v, cfg)\n                overlap, share = evaluate(rect, edge, w, cfg)\n                x0, x1, y0, y1 = rect\n                rows.append({\n                    \"variant\": v[\"variant\"], \"position\": v[\"position\"], \"orientation\": v[\"orientation\"],\n                    \"edge_proxy\": edge, \"w_m\": w,\n                    \"corridor_overlap_m2\": round(overlap, 4), \"band_share_pct\": round(share, 2),\n                    \"seat_area_m2\": round((x1 - x0) * (y1 - y0), 4),\n                    \"inside_window\": x0 >= -EPS and y0 >= -EPS and x1 <= cfg[\"window_m\"] + EPS and y1 <= cfg[\"window_m\"] + EPS,\n                    \"geometry\": \"conflict\" if overlap > EPS else \"clear\",\n                    \"passes_rule\": passes(overlap, share, cfg),\n                })\n    return rows\n\n\ndef variant_summary(rows, cfg):\n    out = []\n    for vid in sorted({r[\"variant\"] for r in rows}):\n        mine = [r for r in rows if r[\"variant\"] == vid]\n        rec = {\"variant\": vid}\n        for edge in cfg[\"edge_proxies\"]:\n            e = [r for r in mine if r[\"edge_proxy\"] == edge]\n            rec[f\"{edge}_share_min_pct\"] = min(r[\"band_share_pct\"] for r in e)\n            rec[f\"{edge}_share_max_pct\"] = max(r[\"band_share_pct\"] for r in e)\n            rec[f\"{edge}_max_overlap_m2\"] = max(r[\"corridor_overlap_m2\"] for r in e)\n            rec[f\"{edge}_passes_all_widths\"] = all(r[\"passes_rule\"] for r in e)\n        rec[\"passes_all_36_conditions\"] = all(r[\"passes_rule\"] for r in mine)\n        out.append(rec)\n    return out\n\n\ndef position_sweep(cfg, orientation=\"parallel\", edge=\"east\", start=1.5, stop=4.5):\n    \"\"\"Move one seat in steps along x and test every tested width (exploratory).\"\"\"\n    step = cfg[\"sweep_step_m\"]\n    out, n = [], int(round((stop - start) / step))\n    for k in range(n + 1):\n        xc = round(start + k * step, 4)\n        rect = seat_rect({\"orientation\": orientation, \"centre_from_east_m\": xc}, cfg)\n        res = [evaluate(rect, edge, w, cfg) for w in cfg[\"widths_m\"]]\n        out.append({\"orientation\": orientation, \"edge_proxy\": edge, \"centre_from_east_m\": xc,\n                    \"share_min_pct\": round(min(s for _, s in res), 2),\n                    \"max_overlap_m2\": round(max(o for o, _ in res), 4),\n                    \"passes_all_widths\": all(passes(o, s, cfg) for o, s in res)})\n    return out\n\n\ndef width_curve(variant, cfg, edge=\"east\", lo=0.5, hi=2.1, step=0.01):\n    out, n = [], int(round((hi - lo) / step))\n    rect = seat_rect(variant, cfg)\n    for k in range(n + 1):\n        w = round(lo + k * step, 4)\n        overlap, share = evaluate(rect, edge, w, cfg)\n        out.append({\"variant\": variant[\"variant\"], \"edge_proxy\": edge, \"w_m\": w,\n                    \"band_share_pct\": round(share, 3), \"corridor_overlap_m2\": round(overlap, 4),\n                    \"passes_rule\": passes(overlap, share, cfg)})\n    return out\n\n\ndef pass_range(curve):\n    ws = [r[\"w_m\"] for r in curve if r[\"passes_rule\"]]\n    return (min(ws), max(ws)) if ws else (None, None)\n\n\ndef internal_checks(cfg, variants, rows):\n    \"\"\"25 checks. Expected values were written by hand before the code was run.\"\"\"\n    V = {v[\"variant\"]: v for v in variants}\n    get = lambda vid, edge, w: next(r for r in rows if r[\"variant\"] == vid and r[\"edge_proxy\"] == edge and abs(r[\"w_m\"] - w) < EPS)\n    checks = []\n\n    def add(name, expected, observed, ok):\n        checks.append({\"n\": len(checks) + 1, \"check\": name, \"expected\": expected, \"observed\": observed,\n                       \"status\": \"PASS\" if ok else \"FAIL\"})\n\n    add(\"Scenario count\", \"6 x 3 x 2 = 36\", len(rows), len(rows) == 36)\n    for vid in sorted(V):\n        a = get(vid, \"east\", 1.0)[\"seat_area_m2\"]\n        add(f\"{vid} seat area\", \"2.00 m2\", f\"{a:.2f} m2\", abs(a - 2.0) < 1e-6)\n    for vid in sorted(V):\n        ins = get(vid, \"east\", 1.0)[\"inside_window\"]\n        add(f\"{vid} inside Window C\", \"True\", str(ins), ins)\n    known = [(\"S1\", \"east\", 1.0, \"band_share_pct\", 80.0, \"seat x 1.7-2.7 in band 1.3-2.5: 0.8 of 1.0\"),\n             (\"S2\", \"east\", 1.0, \"band_share_pct\", 60.0, \"seat x 1.2-3.2 in band 1.3-2.5: 1.2 of 2.0\"),\n             (\"S1\", \"east\", 0.75, \"band_share_pct\", 55.0, \"band 1.05-2.25: 0.55 of 1.0\"),\n             (\"S1\", \"east\", 1.5, \"band_share_pct\", 90.0, \"band 1.8-3.0: 0.9 of 1.0\"),\n             (\"S2\", \"east\", 1.5, \"corridor_overlap_m2\", 0.30, \"seat x 1.2-1.5 inside corridor: 0.3 m x 1.0 m\"),\n             (\"S1\", \"north\", 1.0, \"band_share_pct\", 25.0, \"seat y 2.0-4.0 in band 1.3-2.5: 0.5 of 2.0\"),\n             (\"S4\", \"east\", 1.0, \"band_share_pct\", 15.0, \"seat x 2.2-4.2 in band 1.3-2.5: 0.3 of 2.0\")]\n    for vid, edge, w, col, exp, why in known:\n        obs = get(vid, edge, w)[col]\n        add(f\"Known case {vid} / {edge} / w={w:.2f}: {col}\", f\"{exp} ({why})\", obs, abs(obs - exp) < 1e-6)\n    shares = [r[\"band_share_pct\"] for r in rows]\n    add(\"All band shares within 0-100 %\", \"True\", f\"{min(shares)} to {max(shares)}\", min(shares) >= 0 and max(shares) <= 100)\n    ovs = [r[\"corridor_overlap_m2\"] for r in rows]\n    add(\"All overlaps within 0-2.00 m2\", \"True\", f\"{min(ovs)} to {max(ovs)}\", min(ovs) >= 0 and max(ovs) <= 2.0)\n    none075 = all(r[\"corridor_overlap_m2\"] == 0 for r in rows if r[\"edge_proxy\"] == \"east\" and abs(r[\"w_m\"] - 0.75) < EPS)\n    add(\"No east-proxy overlap at w = 0.75 m\", \"True\", str(none075), none075)\n    sweep_pass = [r[\"centre_from_east_m\"] for r in position_sweep(cfg) if r[\"passes_all_widths\"]]\n    add(\"Position sweep (parallel, east): passing centres\", \"2.00 to 2.15 m\",\n        f\"{min(sweep_pass):.2f} to {max(sweep_pass):.2f} m\" if sweep_pass else \"none\",\n        bool(sweep_pass) and abs(min(sweep_pass) - 2.0) < 1e-6 and abs(max(sweep_pass) - 2.15) < 1e-6)\n    lo, hi = pass_range(width_curve(V[\"S1\"], cfg))\n    add(\"S1 east-proxy pass range of w\", \"0.80 to 1.70 m\", f\"{lo:.2f} to {hi:.2f} m\",\n        lo is not None and abs(lo - 0.80) < 1e-6 and abs(hi - 1.70) < 1e-6)\n    return checks\n\n\ndef grid_cross_check(cfg, variants, rows, cell=0.01):\n    \"\"\"Independent area check that needs no GIS library.\n\n    Re-measures every one of the 36 scenarios by counting 1 cm grid cells (numpy), instead of the\n    interval arithmetic used by evaluate(). All A3 boundaries lie on multiples of 0.05 m, so the\n    count should match exactly. Runs identically on a local machine and in Colab.\n    \"\"\"\n    import numpy as np\n    W = cfg[\"window_m\"]\n    n = int(round(W / cell))\n    centres = (np.arange(n) + 0.5) * cell\n    X, Y = np.meshgrid(centres, centres)          # X = distance from east edge, Y = from north edge\n    cell_area = cell * cell\n    worst_ov, worst_sh, s2 = 0.0, 0.0, None\n    for r in rows:\n        v = next(v for v in variants if v[\"variant\"] == r[\"variant\"])\n        x0, x1, y0, y1 = seat_rect(v, cfg)\n        seat = (X > x0) & (X < x1) & (Y > y0) & (Y < y1)\n        d = X if r[\"edge_proxy\"] == \"east\" else Y\n        w = r[\"w_m\"]\n        ov = float((seat & (d < w)).sum() * cell_area)\n        band = (seat & (d > min(w + cfg[\"c1_m\"], W)) & (d < min(w + cfg[\"c2_m\"], W))).sum()\n        sh = float(100.0 * band / seat.sum())\n        worst_ov = max(worst_ov, abs(ov - r[\"corridor_overlap_m2\"]))\n        worst_sh = max(worst_sh, abs(sh - r[\"band_share_pct\"]))\n        if r[\"variant\"] == \"S2\" and r[\"edge_proxy\"] == \"east\" and abs(w - 1.5) < EPS:\n            s2 = ov\n    ok = worst_ov < 1e-6 and worst_sh < 1e-6 and s2 is not None and abs(s2 - 0.30) < 1e-6\n    return {\"status\": \"PASS\" if ok else \"FAIL\",\n            \"detail\": f\"all 36 scenarios re-measured on a 1 cm grid; S2 / east / w=1.50 m overlap = {s2:.2f} m2; \"\n                      f\"max difference: overlap {worst_ov:.1e} m2, share {worst_sh:.1e} %\"}\n\n\ndef write_csv(path, rows):\n    with open(path, \"w\", encoding=\"utf8\", newline=\"\") as f:\n        wr = csv.DictWriter(f, fieldnames=list(rows[0].keys()))\n        wr.writeheader()\n        wr.writerows(rows)\n", "a3src/a4_window_size.py": "\"\"\"A4 window-size test (instructor feedback on A3, Nov 2026).\n\nQuestion: is the A3 result driven by the assumed 6 x 6 m Window C? For example, what if the\nwindow is 5 x 5 m and the overall geometry is kept?\n\nTwo ways of \"keeping the geometry\" are tested, because they answer different questions:\n  proportional : near / centre / far stay at the same fraction of the window\n                 (2.2/6, 3.2/6, 4.3/6 of W from the east edge; seat centred W/2 from the north edge).\n  fixed_offset : the seat keeps its distance in metres from the edges (2.2 / 3.2 / 4.3 m from east,\n                 3.0 m from north); a seat that no longer fits inside the window is reported, not scored.\n\nThen the A3 sweep result is re-expressed as an offset rule: a seat parallel to the movement edge with\nits centre 2.00-2.15 m from that edge. It is tested for every window size, and also under the north\nproxy while still parallel to the east edge (the limit shown on the Object Card).\n\nUses the same evaluate() and rule as A3; no new thresholds.\n\"\"\"\nimport copy\n\nfrom . import a3_workflow as wf\n\nEPS = wf.EPS\nA3_WINDOW = 6.0\n\n\ndef window_cfg(cfg, W, rule):\n    c = copy.deepcopy(cfg)\n    c[\"window_m\"] = W\n    c[\"seat_centre_from_north_m\"] = W / 2 if rule == \"proportional\" else cfg[\"seat_centre_from_north_m\"]\n    return c\n\n\ndef window_variants(variants, W, rule):\n    out = []\n    for v in variants:\n        v2 = dict(v)\n        if rule == \"proportional\":\n            v2[\"centre_from_east_m\"] = v[\"centre_from_east_m\"] * W / A3_WINDOW\n        out.append(v2)\n    return out\n\n\ndef fits(rect, W):\n    x0, x1, y0, y1 = rect\n    return x0 >= -EPS and y0 >= -EPS and x1 <= W + EPS and y1 <= W + EPS\n\n\ndef run_window_sizes(cfg, variants):\n    rows = []\n    for rule in cfg[\"window_rules\"]:\n        for W in cfg[\"window_sizes_m\"]:\n            c = window_cfg(cfg, W, rule)\n            for v in window_variants(variants, W, rule):\n                rect = wf.seat_rect(v, c)\n                inside = fits(rect, W)\n                for edge in cfg[\"edge_proxies\"]:\n                    for w in cfg[\"widths_m\"]:\n                        ov, sh = wf.evaluate(rect, edge, w, c)\n                        rows.append({\n                            \"rule\": rule, \"window_m\": W, \"variant\": v[\"variant\"], \"orientation\": v[\"orientation\"],\n                            \"centre_from_east_m\": round(v[\"centre_from_east_m\"], 4),\n                            \"centre_from_north_m\": round(c[\"seat_centre_from_north_m\"], 4),\n                            \"edge_proxy\": edge, \"w_m\": w, \"fits_window\": inside,\n                            \"corridor_overlap_m2\": round(ov, 4) if inside else \"\",\n                            \"band_share_pct\": round(sh, 2) if inside else \"\",\n                            \"passes_rule\": bool(inside and wf.passes(ov, sh, c)),\n                        })\n    return rows\n\n\ndef window_summary(rows, cfg):\n    \"\"\"Per rule x window x variant: how many of the 3 east-proxy widths pass, and of all 6 conditions.\"\"\"\n    out = []\n    keys = sorted({(r[\"rule\"], r[\"window_m\"], r[\"variant\"]) for r in rows},\n                  key=lambda k: (cfg[\"window_rules\"].index(k[0]), k[1], k[2]))\n    for rule, W, vid in keys:\n        mine = [r for r in rows if r[\"rule\"] == rule and r[\"window_m\"] == W and r[\"variant\"] == vid]\n        east = [r for r in mine if r[\"edge_proxy\"] == \"east\"]\n        base = next(r for r in east if abs(r[\"w_m\"] - cfg[\"base_width_m\"]) < EPS)\n        out.append({\n            \"rule\": rule, \"window_m\": W, \"variant\": vid, \"fits_window\": mine[0][\"fits_window\"],\n            \"east_base_share_pct\": base[\"band_share_pct\"],\n            \"east_widths_passed_of_3\": sum(r[\"passes_rule\"] for r in east),\n            \"all_conditions_passed_of_6\": sum(r[\"passes_rule\"] for r in mine),\n        })\n    return out\n\n\ndef offset_rule(cfg):\n    \"\"\"Seat parallel to a given edge, centre d from it, tested for every window size.\n\n    'east'           : seat parallel to the east edge (long side N-S), scored with the east proxy.\n    'north_turned'   : seat turned parallel to the north edge (long side E-W), scored with the north proxy.\n    'north_unturned' : seat still parallel to the east edge, scored with the north proxy (the limit).\n    \"\"\"\n    rows = []\n    for W in cfg[\"window_sizes_m\"]:\n        c = window_cfg(cfg, W, \"proportional\")\n        for d in cfg[\"offset_rule_centres_m\"]:\n            cases = {\n                \"east\": (\"east\", (d - 0.5, d + 0.5, W / 2 - 1.0, W / 2 + 1.0)),\n                \"north_turned\": (\"north\", (W / 2 - 1.0, W / 2 + 1.0, d - 0.5, d + 0.5)),\n                \"north_unturned\": (\"north\", (d - 0.5, d + 0.5, W / 2 - 1.0, W / 2 + 1.0)),\n            }\n            for case, (edge, rect) in cases.items():\n                res = [wf.evaluate(rect, edge, w, c) for w in cfg[\"widths_m\"]]\n                ok = fits(rect, W)\n                rows.append({\n                    \"case\": case, \"window_m\": W, \"centre_from_edge_m\": d, \"edge_proxy\": edge, \"fits_window\": ok,\n                    \"share_min_pct\": round(min(s for _, s in res), 2),\n                    \"max_overlap_m2\": round(max(o for o, _ in res), 4),\n                    \"widths_passed_of_3\": sum(ok and wf.passes(o, s, c) for o, s in res),\n                })\n    return rows\n\n\ndef window_checks(cfg, variants, a3_rows, rows, offs):\n    \"\"\"Expected values written by hand before running (A4).\"\"\"\n    checks = []\n\n    def add(name, expected, observed, ok):\n        checks.append({\"n\": f\"W{len(checks) + 1}\", \"check\": name, \"expected\": expected, \"observed\": observed,\n                       \"status\": \"PASS\" if ok else \"FAIL\"})\n\n    def get(rule, W, vid, edge, w):\n        return next(r for r in rows if r[\"rule\"] == rule and abs(r[\"window_m\"] - W) < EPS and r[\"variant\"] == vid\n                    and r[\"edge_proxy\"] == edge and abs(r[\"w_m\"] - w) < EPS)\n\n    n = len(cfg[\"window_rules\"]) * len(cfg[\"window_sizes_m\"]) * len(variants) * len(cfg[\"edge_proxies\"]) * len(cfg[\"widths_m\"])\n    add(\"Window-size row count\", f\"2 rules x {len(cfg['window_sizes_m'])} sizes x 6 x 2 x 3 = {n}\", len(rows), len(rows) == n)\n    for rule in cfg[\"window_rules\"]:\n        diff = 0.0\n        for a in a3_rows:\n            r = get(rule, 6.0, a[\"variant\"], a[\"edge_proxy\"], a[\"w_m\"])\n            diff = max(diff, abs(r[\"band_share_pct\"] - a[\"band_share_pct\"]), abs(r[\"corridor_overlap_m2\"] - a[\"corridor_overlap_m2\"]))\n        add(f\"{rule} at W = 6.0 m reproduces the 36 A3 scenarios\", \"max difference 0\", f\"{diff:.4f}\", diff < 1e-6)\n    r = get(\"proportional\", 5.0, \"S1\", \"east\", 1.5)\n    add(\"5 x 5 proportional: S1 / east / w=1.50 overlap\", \"0.33 m2 (seat x 1.333-2.333 vs corridor 0-1.5: 0.167 x 2.0)\",\n        r[\"corridor_overlap_m2\"], abs(r[\"corridor_overlap_m2\"] - 0.3333) < 1e-3)\n    r = get(\"proportional\", 5.0, \"S1\", \"east\", 1.0)\n    add(\"5 x 5 proportional: S1 / east / w=1.00 band share\", \"100 % (seat 1.333-2.333 inside band 1.3-2.5)\",\n        r[\"band_share_pct\"], abs(r[\"band_share_pct\"] - 100.0) < 1e-6)\n    r = get(\"proportional\", 5.0, \"S1\", \"east\", 0.75)\n    add(\"5 x 5 proportional: S1 / east / w=0.75 band share\", \"91.67 % (band 1.05-2.25: 0.917 of 1.0)\",\n        r[\"band_share_pct\"], abs(r[\"band_share_pct\"] - 91.67) < 0.01)\n    r = get(\"fixed_offset\", 5.0, \"S6\", \"east\", 1.0)\n    add(\"5 x 5 fixed offset: S6 fits the window\", \"False (seat x 3.3-5.3 > 5.0)\", str(r[\"fits_window\"]), r[\"fits_window\"] is False)\n    r = get(\"fixed_offset\", 5.0, \"S1\", \"east\", 0.75)\n    add(\"5 x 5 fixed offset: S1 / east / w=0.75 unchanged from A3\", \"55 %\", r[\"band_share_pct\"], abs(r[\"band_share_pct\"] - 55.0) < 1e-6)\n    east = [o for o in offs if o[\"case\"] == \"east\"]\n    add(\"Offset rule (east, centre 2.00-2.15 m): passes 3/3 widths at every window size\",\n        \"True for all sizes\", str(all(o[\"widths_passed_of_3\"] == 3 for o in east)), all(o[\"widths_passed_of_3\"] == 3 for o in east))\n    turned = [o for o in offs if o[\"case\"] == \"north_turned\"]\n    same = all(abs(t[\"share_min_pct\"] - e[\"share_min_pct\"]) < 1e-6 and t[\"widths_passed_of_3\"] == e[\"widths_passed_of_3\"]\n               for t, e in zip(turned, east))\n    add(\"Offset rule turned to the north edge gives the same result (symmetry)\", \"True\", str(same), same)\n    # First run (26 Nov 2026) expected 0 of 3 at every size and FAILED: in windows <= 5.5 m the seat sits\n    # closer to the north edge, so it partly enters the north band. Replaced by two hand-calculated cases.\n    un = {round(o[\"window_m\"], 2): o for o in offs if o[\"case\"] == \"north_unturned\" and abs(o[\"centre_from_edge_m\"] - 2.0) < EPS}\n    add(\"Offset rule NOT turned, north proxy, W = 6.0 m (centre 2.00 m)\", \"0 of 3 (seat y 2.0-4.0: 12.5 % / 25 % / 50 % at w=0.75/1.00/1.50)\",\n        un[6.0][\"widths_passed_of_3\"], un[6.0][\"widths_passed_of_3\"] == 0)\n    add(\"Offset rule NOT turned, north proxy, W = 5.0 m (centre 2.00 m)\", \"1 of 3 (seat y 1.5-3.5: 37.5 % / 50 % / 60 % at w=0.75/1.00/1.50)\",\n        un[5.0][\"widths_passed_of_3\"], un[5.0][\"widths_passed_of_3\"] == 1)\n    perp = [r for r in wf.position_sweep(cfg, orientation=\"perpendicular\", start=1.0, stop=5.0) if r[\"passes_all_widths\"]]\n    add(\"Perpendicular seat, east proxy, centre swept 1.00-5.00 m: positions passing all 3 widths\",\n        \"none (band 1.2 m deep, seat 2.0 m: 60 % at w=0.75 needs the seat to start <= 1.05 m, inside the w=1.50 corridor)\",\n        len(perp), len(perp) == 0)\n    return checks\n", "a3src/figures.py": "\"\"\"All figures. Every value drawn is read from the rows computed by run_all.py (no typed numbers).\"\"\"\nimport matplotlib\n\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt  # noqa: E402\nfrom matplotlib.patches import Rectangle  # noqa: E402\n\nfrom . import a3_workflow as wf  # noqa: E402\n\nGREEN, TEAL, TAN, ORANGE, RULE = \"#1f5a4e\", \"#9fbfb0\", \"#dcc3a5\", \"#c8623f\", \"#d9532b\"\nINK, MUTED = \"#1f3b34\", \"#5f786f\"\nplt.rcParams.update({\"font.family\": \"DejaVu Sans\", \"axes.titlesize\": 12})\n\n\ndef _get(rows, vid, edge, w):\n    return next(r for r in rows if r[\"variant\"] == vid and r[\"edge_proxy\"] == edge and abs(r[\"w_m\"] - w) < 1e-9)\n\n\ndef a3_figures(out, cfg, variants, rows, curves):\n    W, BW, C1, C2, S = cfg[\"window_m\"], cfg[\"base_width_m\"], cfg[\"c1_m\"], cfg[\"c2_m\"], cfg[\"s_pct\"]\n    fig, axes = plt.subplots(2, 3, figsize=(13.6, 9.4))\n    for ax, v in zip(axes.flat, variants):\n        r = _get(rows, v[\"variant\"], \"east\", BW)\n        x0, x1, y0, y1 = wf.seat_rect(v, cfg)\n        ax.add_patch(Rectangle((-W, -W), W, W, fill=False, ec=\"#4b6b62\", ls=\"--\", lw=1))\n        ax.add_patch(Rectangle((-BW, -W), BW, W, fc=TAN, ec=\"none\", alpha=0.65))\n        ax.add_patch(Rectangle((-(BW + C2), -W), C2 - C1, W, fc=TEAL, ec=\"none\", alpha=0.5))\n        ax.plot([0, 0], [0, -W], color=GREEN, lw=3)\n        conflict = r[\"corridor_overlap_m2\"] > 0\n        ax.add_patch(Rectangle((-x1, -y1), x1 - x0, y1 - y0, fc=ORANGE, ec=\"#8f1d1d\" if conflict else ORANGE,\n                               lw=2.5 if conflict else 1.5, alpha=0.7))\n        ax.set_title(f'{v[\"variant\"]} · {v[\"position\"].lower()} / {v[\"orientation\"]}\\n'\n                     f'Band share {r[\"band_share_pct\"]:.0f}% · overlap {r[\"corridor_overlap_m2\"]:.2f} m²', fontsize=11, color=INK)\n        ax.set_xlim(-W - 0.5, 0.45); ax.set_ylim(-W - 0.5, 0.4); ax.set_aspect(\"equal\")\n        ax.set_xlabel(\"Local x (m) · east edge at x=0\", fontsize=8.5); ax.set_ylabel(\"Local y (m) · north edge at y=0\", fontsize=8.5)\n        ax.grid(alpha=0.12)\n    fig.suptitle(f\"Window C / six ALTERNATIVE placements of ONE 2 × 1 m seat\\nAssumed {W:g} × {W:g} m geometry; east edge proxy, w = {BW:.1f} m\",\n                 fontsize=15, fontweight=\"bold\", color=GREEN)\n    fig.text(0.5, 0.015, \"Schematic geometry only • green = proposed flow-adjacent band • tan = assumed corridor • orange = seat\",\n             ha=\"center\", fontsize=9.5, color=MUTED)\n    fig.tight_layout(rect=(0, 0.03, 1, 0.93), h_pad=2.2)\n    fig.savefig(out / \"fig_six_placements.png\", dpi=160); plt.close(fig)\n\n    ids = [v[\"variant\"] for v in variants]\n    for edge in cfg[\"edge_proxies\"]:\n        base = [_get(rows, v, edge, BW) for v in ids]\n        shares = [r[\"band_share_pct\"] for r in base]\n        lead = ids[max(range(len(ids)), key=lambda i: shares[i])]\n        fig, ax = plt.subplots(figsize=(10, 4.9))\n        ax.bar(ids, shares, color=[GREEN if r[\"passes_rule\"] else \"#5f7d73\" for r in base], width=0.66)\n        for i, s in enumerate(shares):\n            ax.text(i, s + 1.5, f\"{s:.0f}%\", ha=\"center\", fontsize=10, color=\"#2a3d38\")\n        ax.axhline(S, color=RULE, lw=2, ls=\"--\", label=f\"Assumed research threshold s = {S:.0f}%\")\n        ax.set_ylim(0, 105); ax.set_ylabel(\"Proposed flow-adjacent band share (%)\")\n        title = (f\"Base-case {lead} leads; proxy threshold is NOT a comfort standard\" if max(shares) >= S\n                 else f\"No variant reaches s = {S:.0f}% under the {edge} proxy\")\n        ax.set_title(title, fontweight=\"bold\", color=GREEN)\n        ax.legend(frameon=False, loc=\"upper right\", fontsize=9); ax.spines[[\"top\", \"right\"]].set_visible(False)\n        fig.text(0.5, 0.02, f\"{edge.capitalize()} edge proxy · corridor half-width w = {BW:.2f} m · c₁={C1:.2f}m, c₂={C2:.2f}m\",\n                 ha=\"center\", fontsize=9, color=MUTED)\n        fig.tight_layout(rect=(0, 0.05, 1, 1))\n        fig.savefig(out / f\"fig_base_comparison_{edge}.png\", dpi=160); plt.close(fig)\n\n    lo, hi = wf.pass_range(curves[\"S1\"])\n    fig, ax = plt.subplots(figsize=(10, 5.2))\n    if lo is not None:\n        ax.axvspan(lo, hi, color=\"#e7efea\", zorder=0)\n        ax.axvline(lo, color=\"#b08a3c\", lw=1, ls=\":\"); ax.axvline(hi, color=\"#b08a3c\", lw=1, ls=\":\")\n        ax.text(lo + 0.02, 6, \"S1 conditional pass range\\n(excluding geometric overlaps)\", fontsize=8.5, color=GREEN)\n    for vid, col in ((\"S1\", GREEN), (\"S2\", ORANGE)):\n        ax.plot([r[\"w_m\"] for r in curves[vid]], [r[\"band_share_pct\"] for r in curves[vid]], color=col, lw=2.2, label=f\"{vid} band share\")\n        for w in cfg[\"widths_m\"]:\n            ax.plot(w, _get(rows, vid, \"east\", w)[\"band_share_pct\"], \"o\", color=col, ms=6)\n    ax.axhline(S, color=\"#a8522f\", lw=1.6, ls=\"--\", label=f\"Assumed s = {S:.0f}%\")\n    ax.set_xlim(0.5, 2.1); ax.set_ylim(0, 108)\n    ax.set_xlabel(\"Assumed corridor half-width w (m)\"); ax.set_ylabel(\"Proposed band share (%)\")\n    ax.set_title(f\"Sensitivity: S1 clears s only for {lo:.2f} ≤ w ≤ {hi:.2f} m (east proxy)\" if lo is not None\n                 else \"Sensitivity: S1 never clears s (east proxy)\", fontweight=\"bold\", color=GREEN)\n    ax.legend(frameon=False, loc=\"upper left\", fontsize=9); ax.spines[[\"top\", \"right\"]].set_visible(False)\n    fig.tight_layout(); fig.savefig(out / \"fig_width_sensitivity.png\", dpi=160); plt.close(fig)\n\n\nSHADE = {3: (GREEN, \"white\"), 2: (\"#7fa898\", INK), 1: (\"#cfe0d6\", INK), 0: (\"#f3f1ea\", \"#8a8a80\")}\n\n\ndef window_size_figure(out, cfg, variants, wsum, offs):\n    \"\"\"KEY FIGURE (Object Card). Rows: six labelled seats with proportional geometry, the offset rule,\n    and the limit (same seat scored with the north proxy). Cells: east-proxy widths passed of 3.\"\"\"\n    sizes = cfg[\"window_sizes_m\"]\n    vlabel = {v[\"variant\"]: f'{v[\"variant\"]} · {v[\"position\"].lower()} · {\"parallel\" if v[\"orientation\"] == \"parallel\" else \"perpendicular\"}'\n              for v in variants}\n    fig = plt.figure(figsize=(12.4, 7.3))\n    ax = fig.add_axes((0.30, 0.17, 0.66, 0.66))\n    nrows = len(variants) + 2\n    ax.set_xlim(0, len(sizes)); ax.set_ylim(nrows + 0.35, -0.05); ax.axis(\"off\")\n\n    def cell(col, row, k, sub=None, note=None):\n        if k is None:\n            ax.add_patch(Rectangle((col + 0.04, row + 0.06), 0.92, 0.88, fc=\"white\", ec=\"#c9c2b4\", hatch=\"///\", lw=0.8))\n            ax.text(col + 0.5, row + 0.5, \"does not fit\", ha=\"center\", va=\"center\", fontsize=8, color=\"#8a8a80\")\n            return\n        fc, tc = SHADE[k]\n        ax.add_patch(Rectangle((col + 0.04, row + 0.06), 0.92, 0.88, fc=fc, ec=\"white\", lw=0))\n        ax.text(col + 0.5, row + (0.42 if sub else 0.5), f\"{k}/3\", ha=\"center\", va=\"center\", fontsize=13, fontweight=\"bold\", color=tc)\n        if sub:\n            ax.text(col + 0.5, row + 0.76, sub, ha=\"center\", va=\"center\", fontsize=8, color=tc if k == 3 else MUTED)\n\n    for j, W in enumerate(sizes):\n        tag = \"  A3\" if abs(W - 6.0) < 1e-9 else (\"  5 × 5 asked\" if abs(W - 5.0) < 1e-9 else \"\")\n        ax.text(j + 0.5, -0.32, f\"{W:g} × {W:g} m\", ha=\"center\", va=\"bottom\", fontsize=10.5, fontweight=\"bold\", color=INK)\n        if tag:\n            ax.text(j + 0.5, -0.62, tag.strip(), ha=\"center\", va=\"bottom\", fontsize=8.5, color=RULE, fontweight=\"bold\")\n            ax.add_patch(Rectangle((j + 0.01, -0.05), 0.98, nrows + 0.35, fill=False, ec=RULE, lw=1.6, ls=\"--\"))\n    for i, v in enumerate(variants):\n        fig.text(0.295, 0.83 - 0.66 * (i + 0.5) / (nrows + 0.4), vlabel[v[\"variant\"]], ha=\"right\", va=\"center\", fontsize=10.5, color=INK)\n        for j, W in enumerate(sizes):\n            s = next(x for x in wsum if x[\"rule\"] == \"proportional\" and abs(x[\"window_m\"] - W) < 1e-9 and x[\"variant\"] == v[\"variant\"])\n            centre = v[\"centre_from_east_m\"] * W / 6.0\n            cell(j, i, s[\"east_widths_passed_of_3\"] if s[\"fits_window\"] else None, f\"centre {centre:.2f} m\")\n    r_off, r_lim = len(variants) + 0.25, len(variants) + 1.25\n    ax.plot([0, len(sizes)], [len(variants) + 0.12] * 2, color=INK, lw=1)\n    lab = [(\"Rule: parallel seat, centre\\n2.00–2.15 m from the movement edge\", r_off, \"east\", GREEN),\n           (\"Limit: same seat if movement runs\\nalong the north edge (not turned)\", r_lim, \"north_unturned\", RULE)]\n    for text, r, case, col in lab:\n        fig.text(0.295, 0.83 - 0.66 * (r + 0.5) / (nrows + 0.4), text, ha=\"right\", va=\"center\", fontsize=10.5, color=col, fontweight=\"bold\")\n        for j, W in enumerate(sizes):\n            ks = [o[\"widths_passed_of_3\"] for o in offs if o[\"case\"] == case and abs(o[\"window_m\"] - W) < 1e-9]\n            k = min(ks)\n            cell(j, r, k, \"all 4 centres\" if min(ks) == max(ks) else f\"{min(ks)}–{max(ks)} of 3\")\n    fig.text(0.03, 0.955, \"The winning label changes with window size; a 2.00–2.15 m edge offset always passes\",\n             fontsize=16, fontweight=\"bold\", color=GREEN)\n    fig.text(0.03, 0.915, \"Cells: tested corridor half-widths passed (w = 0.75 / 1.00 / 1.50 m), east edge as the assumed movement edge. \"\n             \"Labelled seats keep their place as a fraction of the window.\", fontsize=9.5, color=MUTED)\n    fig.text(0.03, 0.06, f\"Pass = 0 m² corridor overlap AND ≥ {cfg['s_pct']:.0f} % of the 2.0 × 1.0 m seat inside the band from w + {cfg['c1_m']:.2f} m \"\n             f\"to w + {cfg['c2_m']:.2f} m (rule set in A3, before these runs).\", fontsize=9.5, color=INK)\n    fig.text(0.03, 0.025, \"Schematic geometry; edge, w, c₁, c₂ and s are assumptions, not observed movement or use. \"\n             \"Source: outputs/a4_window_size_summary.csv, outputs/a4_offset_rule.csv.\", fontsize=9, color=MUTED)\n    fig.savefig(out / \"fig_window_size.png\", dpi=200); plt.close(fig)\n\n\ndef window_plans_figure(out, cfg, variants):\n    \"\"\"Three plans at the same scale: A3 S1 (6 x 6), S1 kept proportional in 5 x 5, the offset rule.\"\"\"\n    S1 = next(v for v in variants if v[\"variant\"] == \"S1\")\n    C1, C2 = cfg[\"c1_m\"], cfg[\"c2_m\"]\n    panels = [\n        (\"A3 · 6 × 6 m · S1 centre 2.20 m\", 6.0, S1[\"centre_from_east_m\"], 0.75),\n        (\"5 × 5 m, geometry kept · S1 centre 1.83 m\", 5.0, S1[\"centre_from_east_m\"] * 5.0 / 6.0, 1.5),\n        (\"Offset rule · centre 2.00–2.15 m\", 5.0, None, None),\n    ]\n    fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.7))\n    for ax, (title, W, xc, w) in zip(axes, panels):\n        ax.add_patch(Rectangle((-W, -W), W, W, fc=\"#f7f4ec\", ec=\"#e0602e\", ls=\"--\", lw=1.6))\n        ax.plot([0, 0], [0.3, -W - 0.3], color=\"#1d5a8a\", lw=4)\n        yc = W / 2\n        if xc is not None:\n            rect = wf.seat_rect(S1, dict(cfg, seat_centre_from_north_m=yc), centre_from_east=xc)\n            ov, sh = wf.evaluate(rect, \"east\", w, dict(cfg, window_m=W))\n            ax.add_patch(Rectangle((-w, -W), w, W, fc=TAN, ec=\"none\", alpha=0.75))\n            ax.add_patch(Rectangle((-(w + C2), -W), C2 - C1, W, fc=TEAL, ec=\"none\", alpha=0.55))\n            x0, x1, y0, y1 = rect\n            ax.add_patch(Rectangle((-x1, -y1), x1 - x0, y1 - y0, fc=ORANGE, ec=\"#8f1d1d\", lw=1.5, alpha=0.85))\n            if ov > 1e-9:\n                ax.add_patch(Rectangle((-w, -y1), w - x0, y1 - y0, fc=\"none\", ec=\"#8f1d1d\", hatch=\"xxx\", lw=0))\n            verdict = (f\"w = {w:.2f} m: overlap {ov:.2f} m² → fails\" if ov > 1e-9\n                       else f\"w = {w:.2f} m: band share {sh:.0f} % < {cfg['s_pct']:.0f} % → fails\")\n            ax.text(-W / 2, -W - 0.75, verdict, ha=\"center\", fontsize=10.5, color=\"#8f1d1d\", fontweight=\"bold\")\n        else:\n            for wv, ls in zip(cfg[\"widths_m\"], (\":\", \"--\", \"-\")):\n                ax.plot([-wv, -wv], [0, -W], color=\"#8a6d3b\", lw=1.2, ls=ls)\n            ax.add_patch(Rectangle((-2.65, -yc - 1.0), 1.15, 2.0, fc=ORANGE, alpha=0.25, ec=\"none\"))\n            ax.add_patch(Rectangle((-2.5, -yc - 1.0), 1.0, 2.0, fc=ORANGE, ec=\"#8f1d1d\", lw=1.5, alpha=0.85))\n            ax.annotate(\"\", xy=(-2.075, -yc - 1.35), xytext=(0, -yc - 1.35), arrowprops=dict(arrowstyle=\"<->\", color=INK, lw=1))\n            ax.text(-2.25, -yc - 1.35, \"centre 2.00–2.15 m\", ha=\"right\", va=\"center\", fontsize=9, color=INK)\n            ax.text(-W / 2, -W - 1.25, \"brown lines: corridor edge at w = 0.75 (dotted), 1.00 (dashed), 1.50 m (solid)\",\n                    ha=\"center\", fontsize=8.5, color=\"#8a6d3b\")\n            ax.text(-W / 2, -W - 0.75, \"passes w = 0.75, 1.00 and 1.50 m\", ha=\"center\", fontsize=10.5, color=GREEN, fontweight=\"bold\")\n        ax.set_title(title, fontsize=11.5, color=INK, fontweight=\"bold\")\n        ax.set_xlim(-6.4, 0.9); ax.set_ylim(-7.4, 0.7); ax.set_aspect(\"equal\"); ax.axis(\"off\")\n        ax.text(0.15, -W / 2, \"east edge\\n(assumed\\nmovement)\", fontsize=8, color=\"#1d5a8a\", va=\"center\")\n    fig.suptitle(\"Keeping the geometry moves the seat; the edge offset does not\", fontsize=14.5, fontweight=\"bold\", color=GREEN, x=0.02, ha=\"left\")\n    fig.text(0.02, 0.02, \"Same scale, plan, north up. Tan = corridor at the width shown, green = band, orange = 2.0 × 1.0 m seat. Schematic, not surveyed.\",\n             fontsize=9, color=MUTED)\n    fig.tight_layout(rect=(0, 0.04, 1, 0.95), w_pad=0.5)\n    fig.savefig(out / \"fig_window_plans.png\", dpi=200); plt.close(fig)\n\n\ndef draw_all(out, cfg, variants, rows, curves, wsum, offs):\n    a3_figures(out, cfg, variants, rows, curves)\n    window_size_figure(out, cfg, variants, wsum, offs)\n    window_plans_figure(out, cfg, variants)\n", "run_all.py": "\"\"\"Window C seat placement: run the whole workflow (A3 baseline + A4 window-size test).\n\nUsage:  python run_all.py\nWrites outputs/ (CSV tables, figures, check log, run record) and compares the run with reference/.\nEvery run executes the same steps and the same checks, on a local machine or in Colab; nothing is\noptional or skipped. Exit code 0 = all checks PASS and results REPRODUCED; 1 = something differs.\n\"\"\"\nimport platform\nimport sys\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parent\nsys.path.insert(0, str(ROOT))\nfrom a3src import a3_workflow as wf  # noqa: E402\nfrom a3src import a4_window_size as ws  # noqa: E402\nfrom a3src import figures  # noqa: E402\nimport check_reproduction  # noqa: E402\n\nOUT = ROOT / \"outputs\"\nOUT.mkdir(exist_ok=True)\ncfg, variants = wf.load_inputs(ROOT)\nV = {v[\"variant\"]: v for v in variants}\n\n# 1. A3 baseline: 6 placements x 3 widths x 2 edge proxies = 36 scenarios\nrows = wf.run_scenarios(cfg, variants)\nwf.write_csv(OUT / \"a3_scenarios.csv\", rows)\nsummary = wf.variant_summary(rows, cfg)\nwf.write_csv(OUT / \"a3_variant_summary.csv\", summary)\nsweep = wf.position_sweep(cfg)\nwf.write_csv(OUT / \"a3_position_sweep.csv\", sweep)\ncurves = {vid: wf.width_curve(V[vid], cfg) for vid in (\"S1\", \"S2\")}\nwf.write_csv(OUT / \"a3_width_sensitivity.csv\", curves[\"S1\"] + curves[\"S2\"])\n\n# 2. A4 window-size test: 2 geometry rules x 7 window sizes x 36 = 504 scenarios, plus the offset rule\nwrows = ws.run_window_sizes(cfg, variants)\nwf.write_csv(OUT / \"a4_window_size.csv\", wrows)\nwsum = ws.window_summary(wrows, cfg)\nwf.write_csv(OUT / \"a4_window_size_summary.csv\", wsum)\noffs = ws.offset_rule(cfg)\nwf.write_csv(OUT / \"a4_offset_rule.csv\", offs)\n\n# 3. checks: 25 A3 hand-calculated checks, grid cross-check of all 36 scenarios, A4 window checks\nchecks = wf.internal_checks(cfg, variants, rows)\ngrid = wf.grid_cross_check(cfg, variants, rows)\nwchecks = ws.window_checks(cfg, variants, rows, wrows, offs)\nlog = checks + [{\"n\": \"G\", \"check\": \"Grid cross-check (1 cm cells, numpy) of all 36 scenarios\",\n                 \"expected\": \"same overlap and share as the core; S2 / east / w=1.50 m = 0.30 m2\",\n                 \"observed\": grid[\"detail\"], \"status\": grid[\"status\"]}] + wchecks\nwf.write_csv(OUT / \"a3_checks.csv\", log)\nn_pass = sum(c[\"status\"] == \"PASS\" for c in log)\n\n# 4. figures\nfigures.draw_all(OUT, cfg, variants, rows, curves, wsum, offs)\n\n# 5. reproduction against reference/ (A3 36 scenarios + A4 504 scenarios)\nrepro_ok, repro_msgs = check_reproduction.compare(ROOT)\n\n# 6. run record (versions + results)\nimport matplotlib  # noqa: E402\nimport numpy  # noqa: E402\nstamp = datetime.now(timezone.utc).strftime(\"%Y-%m-%d %H:%M UTC\")\nvers = [f\"python {platform.python_version()}\", f\"numpy {numpy.__version__}\", f\"matplotlib {matplotlib.__version__}\",\n        f\"platform {platform.system()} {platform.machine()}\"]\n(OUT / \"versions.txt\").write_text(\"\\n\".join(vers) + \"\\n\", encoding=\"utf8\")\n\npassing = [r[\"centre_from_east_m\"] for r in sweep if r[\"passes_all_widths\"]]\nfive = [s for s in wsum if s[\"rule\"] == \"proportional\" and abs(s[\"window_m\"] - 5.0) < 1e-9]\nlines = [\n    f\"Run: {stamp}\",\n    \"Environment: \" + \" | \".join(vers),\n    f\"A3 scenarios: {len(rows)} rows -> outputs/a3_scenarios.csv\",\n    f\"A4 window-size scenarios: {len(wrows)} rows -> outputs/a4_window_size.csv\",\n    f\"Checks: {n_pass} of {len(log)} PASS (25 A3 hand-calculated + 1 grid cross-check + {len(wchecks)} A4 window-size)\",\n    f\"Grid cross-check: {grid['status']} - {grid['detail']}\",\n    \"Variants passing all 36 A3 conditions: \" + (\", \".join(s[\"variant\"] for s in summary if s[\"passes_all_36_conditions\"]) or \"none\"),\n    \"A3 sweep (parallel seat, east proxy) passes all three widths at centres: \"\n    + (f\"{min(passing):.2f}-{max(passing):.2f} m\" if passing else \"none\"),\n    \"5 x 5 m window, proportional geometry, east widths passed of 3: \"\n    + \", \".join(f\"{s['variant']} {s['east_widths_passed_of_3']}\" for s in five),\n    \"Offset rule (parallel to east edge, centre 2.00-2.15 m): passes 3 of 3 widths at window sizes \"\n    + \", \".join(f\"{W:g}\" for W in cfg[\"window_sizes_m\"]\n                if all(o[\"widths_passed_of_3\"] == 3 for o in offs if o[\"case\"] == \"east\" and o[\"window_m\"] == W)) + \" m\",\n    \"Reproduction: \" + (\"REPRODUCED - \" if repro_ok else \"NOT REPRODUCED - \") + \"; \".join(repro_msgs[:6]),\n    \"Key figure: outputs/fig_window_size.png (Object Card).\",\n    \"These are geometric screening results under assumed inputs, not evidence of pedestrian flow or comfort.\",\n]\nfor c in log:\n    if c[\"status\"] != \"PASS\":\n        lines.append(f\"  FAIL {c['n']}: {c['check']} | expected {c['expected']} | observed {c['observed']}\")\nverdict = \"ALL CHECKS PASS · REPRODUCED\" if (n_pass == len(log) and repro_ok) else \"CHECK FAILED - see lines above\"\nlines.append(\"RESULT: \" + verdict)\n(OUT / \"run_record.txt\").write_text(\"\\n\".join(lines) + \"\\n\", encoding=\"utf8\")\nprint(\"\\n\".join(lines))\nif verdict != \"ALL CHECKS PASS · REPRODUCED\":\n    sys.exit(1)\n", "check_reproduction.py": "\"\"\"Compare a fresh run with the reference results.\n\nCalled at the end of run_all.py; can also be run on its own after run_all.py:\n    python check_reproduction.py\n\"\"\"\nimport csv\nimport sys\nfrom pathlib import Path\n\nTOL = 0.01\nTABLES = [\n    (\"a3_scenarios.csv\", (\"variant\", \"edge_proxy\", \"w_m\")),\n    (\"a4_window_size.csv\", (\"rule\", \"window_m\", \"variant\", \"edge_proxy\", \"w_m\")),\n]\n\n\ndef _key(r, cols):\n    return tuple(f\"{float(r[c]):.2f}\" if c in (\"w_m\", \"window_m\") else r[c] for c in cols)\n\n\ndef _load(path, cols):\n    with open(path, encoding=\"utf8\", newline=\"\") as f:\n        return {_key(r, cols): r for r in csv.DictReader(f)}\n\n\ndef _num(x):\n    return None if x in (\"\", None) else float(x)\n\n\ndef compare(root):\n    root = Path(root)\n    msgs, ok = [], True\n    for name, cols in TABLES:\n        new = _load(root / \"outputs\" / name, cols)\n        ref = _load(root / \"reference\" / name.replace(\".csv\", \"_reference.csv\"), cols)\n        problems = [] if set(new) == set(ref) else [f\"{name}: scenario keys differ ({len(new)} new vs {len(ref)} reference)\"]\n        for key in sorted(set(new) & set(ref)):\n            for col in (\"corridor_overlap_m2\", \"band_share_pct\"):\n                a, b = _num(new[key][col]), _num(ref[key][col])\n                if (a is None) != (b is None) or (a is not None and abs(a - b) > TOL):\n                    problems.append(f\"{name} {key} {col}: got {new[key][col]}, reference {ref[key][col]}\")\n            if new[key][\"passes_rule\"].strip().lower() != ref[key][\"passes_rule\"].strip().lower():\n                problems.append(f\"{name} {key} passes_rule differs\")\n        if problems:\n            ok = False\n            msgs += problems\n        else:\n            msgs.append(f\"{name}: all {len(ref)} rows match within {TOL}\")\n    return ok, msgs\n\n\nif __name__ == \"__main__\":\n    ok, msgs = compare(Path(__file__).resolve().parent)\n    print((\"REPRODUCED\" if ok else \"NOT REPRODUCED (expected if inputs/ were edited)\") + \":\")\n    for m in msgs:\n        print(\"  -\", m)\n    sys.exit(0 if ok else 1)\n", "inputs/config.json": "{\n  \"frame\": \"local-metre schematic: x = distance inward from the east edge proxy, y = distance south from the north edge proxy\",\n  \"provenance_only\": \"Harbour Road Garden, OpenStreetMap way 27059126 (ID cited for location; no coordinates are loaded)\",\n  \"window_m\": 6.0,\n  \"seat_long_m\": 2.0,\n  \"seat_short_m\": 1.0,\n  \"seat_centre_from_north_m\": 3.0,\n  \"widths_m\": [\n    0.75,\n    1.0,\n    1.5\n  ],\n  \"base_width_m\": 1.0,\n  \"c1_m\": 0.3,\n  \"c2_m\": 1.5,\n  \"s_pct\": 60.0,\n  \"edge_proxies\": [\n    \"east\",\n    \"north\"\n  ],\n  \"sweep_step_m\": 0.05,\n  \"status\": \"Window C, edge proxies, w, c1, c2 and s are researcher assumptions; none is a measured path or a comfort standard. A4 adds the window-size test (instructor feedback): 4.0-7.0 m square windows, geometry kept either proportionally or as fixed offsets.\",\n  \"window_sizes_m\": [\n    4.0,\n    4.5,\n    5.0,\n    5.5,\n    6.0,\n    6.5,\n    7.0\n  ],\n  \"window_rules\": [\n    \"proportional\",\n    \"fixed_offset\"\n  ],\n  \"offset_rule_centres_m\": [\n    2.0,\n    2.05,\n    2.1,\n    2.15\n  ]\n}\n", "inputs/variants.csv": "variant,position,orientation,long_axis,centre_from_east_m\nS1,Near,parallel,N-S,2.2\nS2,Near,perpendicular,E-W,2.2\nS3,Centre,parallel,N-S,3.2\nS4,Centre,perpendicular,E-W,3.2\nS5,Far,parallel,N-S,4.3\nS6,Far,perpendicular,E-W,4.3\n", "reference/a3_scenarios_reference.csv": "variant,position,orientation,edge_proxy,w_m,corridor_overlap_m2,band_share_pct,seat_area_m2,inside_window,geometry,passes_rule\nS1,Near,parallel,east,0.75,0.0,55.0,2.0,True,clear,False\nS2,Near,perpendicular,east,0.75,0.0,52.5,2.0,True,clear,False\nS3,Centre,parallel,east,0.75,0.0,0.0,2.0,True,clear,False\nS4,Centre,perpendicular,east,0.75,0.0,2.5,2.0,True,clear,False\nS5,Far,parallel,east,0.75,0.0,0.0,2.0,True,clear,False\nS6,Far,perpendicular,east,0.75,0.0,0.0,2.0,True,clear,False\nS1,Near,parallel,east,1.0,0.0,80.0,2.0,True,clear,True\nS2,Near,perpendicular,east,1.0,0.0,60.0,2.0,True,clear,True\nS3,Centre,parallel,east,1.0,0.0,0.0,2.0,True,clear,False\nS4,Centre,perpendicular,east,1.0,0.0,15.0,2.0,True,clear,False\nS5,Far,parallel,east,1.0,0.0,0.0,2.0,True,clear,False\nS6,Far,perpendicular,east,1.0,0.0,0.0,2.0,True,clear,False\nS1,Near,parallel,east,1.5,0.0,90.0,2.0,True,clear,True\nS2,Near,perpendicular,east,1.5,0.3,60.0,2.0,True,conflict,False\nS3,Centre,parallel,east,1.5,0.0,30.0,2.0,True,clear,False\nS4,Centre,perpendicular,east,1.5,0.0,40.0,2.0,True,clear,False\nS5,Far,parallel,east,1.5,0.0,0.0,2.0,True,clear,False\nS6,Far,perpendicular,east,1.5,0.0,0.0,2.0,True,clear,False\nS1,Near,parallel,north,0.75,0.0,12.5,2.0,True,clear,False\nS2,Near,perpendicular,north,0.75,0.0,0.0,2.0,True,clear,False\nS3,Centre,parallel,north,0.75,0.0,12.5,2.0,True,clear,False\nS4,Centre,perpendicular,north,0.75,0.0,0.0,2.0,True,clear,False\nS5,Far,parallel,north,0.75,0.0,12.5,2.0,True,clear,False\nS6,Far,perpendicular,north,0.75,0.0,0.0,2.0,True,clear,False\nS1,Near,parallel,north,1.0,0.0,25.0,2.0,True,clear,False\nS2,Near,perpendicular,north,1.0,0.0,0.0,2.0,True,clear,False\nS3,Centre,parallel,north,1.0,0.0,25.0,2.0,True,clear,False\nS4,Centre,perpendicular,north,1.0,0.0,0.0,2.0,True,clear,False\nS5,Far,parallel,north,1.0,0.0,25.0,2.0,True,clear,False\nS6,Far,perpendicular,north,1.0,0.0,0.0,2.0,True,clear,False\nS1,Near,parallel,north,1.5,0.0,50.0,2.0,True,clear,False\nS2,Near,perpendicular,north,1.5,0.0,50.0,2.0,True,clear,False\nS3,Centre,parallel,north,1.5,0.0,50.0,2.0,True,clear,False\nS4,Centre,perpendicular,north,1.5,0.0,50.0,2.0,True,clear,False\nS5,Far,parallel,north,1.5,0.0,50.0,2.0,True,clear,False\nS6,Far,perpendicular,north,1.5,0.0,50.0,2.0,True,clear,False\n", "reference/a4_window_size_reference.csv": "rule,window_m,variant,orientation,centre_from_east_m,centre_from_north_m,edge_proxy,w_m,fits_window,corridor_overlap_m2,band_share_pct,passes_rule\nproportional,4.0,S1,parallel,1.4667,2.0,east,0.75,True,0.0,91.67,True\nproportional,4.0,S1,parallel,1.4667,2.0,east,1.0,True,0.0667,66.67,False\nproportional,4.0,S1,parallel,1.4667,2.0,east,1.5,True,1.0667,16.67,False\nproportional,4.0,S1,parallel,1.4667,2.0,north,0.75,True,0.0,60.0,True\nproportional,4.0,S1,parallel,1.4667,2.0,north,1.0,True,0.0,60.0,True\nproportional,4.0,S1,parallel,1.4667,2.0,north,1.5,True,0.5,60.0,False\nproportional,4.0,S2,perpendicular,1.4667,2.0,east,0.75,True,0.2833,60.0,False\nproportional,4.0,S2,perpendicular,1.4667,2.0,east,1.0,True,0.5333,58.33,False\nproportional,4.0,S2,perpendicular,1.4667,2.0,east,1.5,True,1.0333,33.33,False\nproportional,4.0,S2,perpendicular,1.4667,2.0,north,0.75,True,0.0,75.0,True\nproportional,4.0,S2,perpendicular,1.4667,2.0,north,1.0,True,0.0,100.0,True\nproportional,4.0,S2,perpendicular,1.4667,2.0,north,1.5,True,0.0,70.0,True\nproportional,4.0,S3,parallel,2.1333,2.0,east,0.75,True,0.0,61.67,True\nproportional,4.0,S3,parallel,2.1333,2.0,east,1.0,True,0.0,86.67,True\nproportional,4.0,S3,parallel,2.1333,2.0,east,1.5,True,0.0,83.33,True\nproportional,4.0,S3,parallel,2.1333,2.0,north,0.75,True,0.0,60.0,True\nproportional,4.0,S3,parallel,2.1333,2.0,north,1.0,True,0.0,60.0,True\nproportional,4.0,S3,parallel,2.1333,2.0,north,1.5,True,0.5,60.0,False\nproportional,4.0,S4,perpendicular,2.1333,2.0,east,0.75,True,0.0,55.83,False\nproportional,4.0,S4,perpendicular,2.1333,2.0,east,1.0,True,0.0,60.0,True\nproportional,4.0,S4,perpendicular,2.1333,2.0,east,1.5,True,0.3667,60.0,False\nproportional,4.0,S4,perpendicular,2.1333,2.0,north,0.75,True,0.0,75.0,True\nproportional,4.0,S4,perpendicular,2.1333,2.0,north,1.0,True,0.0,100.0,True\nproportional,4.0,S4,perpendicular,2.1333,2.0,north,1.5,True,0.0,70.0,True\nproportional,4.0,S5,parallel,2.8667,2.0,east,0.75,True,0.0,0.0,False\nproportional,4.0,S5,parallel,2.8667,2.0,east,1.0,True,0.0,13.33,False\nproportional,4.0,S5,parallel,2.8667,2.0,east,1.5,True,0.0,63.33,True\nproportional,4.0,S5,parallel,2.8667,2.0,north,0.75,True,0.0,60.0,True\nproportional,4.0,S5,parallel,2.8667,2.0,north,1.0,True,0.0,60.0,True\nproportional,4.0,S5,parallel,2.8667,2.0,north,1.5,True,0.5,60.0,False\nproportional,4.0,S6,perpendicular,2.8667,2.0,east,0.75,True,0.0,19.17,False\nproportional,4.0,S6,perpendicular,2.8667,2.0,east,1.0,True,0.0,31.67,False\nproportional,4.0,S6,perpendicular,2.8667,2.0,east,1.5,True,0.0,56.67,False\nproportional,4.0,S6,perpendicular,2.8667,2.0,north,0.75,True,0.0,75.0,True\nproportional,4.0,S6,perpendicular,2.8667,2.0,north,1.0,True,0.0,100.0,True\nproportional,4.0,S6,perpendicular,2.8667,2.0,north,1.5,True,0.0,70.0,True\nproportional,4.5,S1,parallel,1.65,2.25,east,0.75,True,0.0,100.0,True\nproportional,4.5,S1,parallel,1.65,2.25,east,1.0,True,0.0,85.0,True\nproportional,4.5,S1,parallel,1.65,2.25,east,1.5,True,0.7,35.0,False\nproportional,4.5,S1,parallel,1.65,2.25,north,0.75,True,0.0,50.0,False\nproportional,4.5,S1,parallel,1.65,2.25,north,1.0,True,0.0,60.0,True\nproportional,4.5,S1,parallel,1.65,2.25,north,1.5,True,0.25,60.0,False\nproportional,4.5,S2,perpendicular,1.65,2.25,east,0.75,True,0.1,60.0,False\nproportional,4.5,S2,perpendicular,1.65,2.25,east,1.0,True,0.35,60.0,False\nproportional,4.5,S2,perpendicular,1.65,2.25,east,1.5,True,0.85,42.5,False\nproportional,4.5,S2,perpendicular,1.65,2.25,north,0.75,True,0.0,50.0,False\nproportional,4.5,S2,perpendicular,1.65,2.25,north,1.0,True,0.0,75.0,True\nproportional,4.5,S2,perpendicular,1.65,2.25,north,1.5,True,0.0,95.0,True\nproportional,4.5,S3,parallel,2.4,2.25,east,0.75,True,0.0,35.0,False\nproportional,4.5,S3,parallel,2.4,2.25,east,1.0,True,0.0,60.0,True\nproportional,4.5,S3,parallel,2.4,2.25,east,1.5,True,0.0,100.0,True\nproportional,4.5,S3,parallel,2.4,2.25,north,0.75,True,0.0,50.0,False\nproportional,4.5,S3,parallel,2.4,2.25,north,1.0,True,0.0,60.0,True\nproportional,4.5,S3,parallel,2.4,2.25,north,1.5,True,0.25,60.0,False\nproportional,4.5,S4,perpendicular,2.4,2.25,east,0.75,True,0.0,42.5,False\nproportional,4.5,S4,perpendicular,2.4,2.25,east,1.0,True,0.0,55.0,False\nproportional,4.5,S4,perpendicular,2.4,2.25,east,1.5,True,0.1,60.0,False\nproportional,4.5,S4,perpendicular,2.4,2.25,north,0.75,True,0.0,50.0,False\nproportional,4.5,S4,perpendicular,2.4,2.25,north,1.0,True,0.0,75.0,True\nproportional,4.5,S4,perpendicular,2.4,2.25,north,1.5,True,0.0,95.0,True\nproportional,4.5,S5,parallel,3.225,2.25,east,0.75,True,0.0,0.0,False\nproportional,4.5,S5,parallel,3.225,2.25,east,1.0,True,0.0,0.0,False\nproportional,4.5,S5,parallel,3.225,2.25,east,1.5,True,0.0,27.5,False\nproportional,4.5,S5,parallel,3.225,2.25,north,0.75,True,0.0,50.0,False\nproportional,4.5,S5,parallel,3.225,2.25,north,1.0,True,0.0,60.0,True\nproportional,4.5,S5,parallel,3.225,2.25,north,1.5,True,0.25,60.0,False\nproportional,4.5,S6,perpendicular,3.225,2.25,east,0.75,True,0.0,1.25,False\nproportional,4.5,S6,perpendicular,3.225,2.25,east,1.0,True,0.0,13.75,False\nproportional,4.5,S6,perpendicular,3.225,2.25,east,1.5,True,0.0,38.75,False\nproportional,4.5,S6,perpendicular,3.225,2.25,north,0.75,True,0.0,50.0,False\nproportional,4.5,S6,perpendicular,3.225,2.25,north,1.0,True,0.0,75.0,True\nproportional,4.5,S6,perpendicular,3.225,2.25,north,1.5,True,0.0,95.0,True\nproportional,5.0,S1,parallel,1.8333,2.5,east,0.75,True,0.0,91.67,True\nproportional,5.0,S1,parallel,1.8333,2.5,east,1.0,True,0.0,100.0,True\nproportional,5.0,S1,parallel,1.8333,2.5,east,1.5,True,0.3333,53.33,False\nproportional,5.0,S1,parallel,1.8333,2.5,north,0.75,True,0.0,37.5,False\nproportional,5.0,S1,parallel,1.8333,2.5,north,1.0,True,0.0,50.0,False\nproportional,5.0,S1,parallel,1.8333,2.5,north,1.5,True,0.0,60.0,True\nproportional,5.0,S2,perpendicular,1.8333,2.5,east,0.75,True,0.0,60.0,True\nproportional,5.0,S2,perpendicular,1.8333,2.5,east,1.0,True,0.1667,60.0,False\nproportional,5.0,S2,perpendicular,1.8333,2.5,east,1.5,True,0.6667,51.67,False\nproportional,5.0,S2,perpendicular,1.8333,2.5,north,0.75,True,0.0,25.0,False\nproportional,5.0,S2,perpendicular,1.8333,2.5,north,1.0,True,0.0,50.0,False\nproportional,5.0,S2,perpendicular,1.8333,2.5,north,1.5,True,0.0,100.0,True\nproportional,5.0,S3,parallel,2.6667,2.5,east,0.75,True,0.0,8.33,False\nproportional,5.0,S3,parallel,2.6667,2.5,east,1.0,True,0.0,33.33,False\nproportional,5.0,S3,parallel,2.6667,2.5,east,1.5,True,0.0,83.33,True\nproportional,5.0,S3,parallel,2.6667,2.5,north,0.75,True,0.0,37.5,False\nproportional,5.0,S3,parallel,2.6667,2.5,north,1.0,True,0.0,50.0,False\nproportional,5.0,S3,parallel,2.6667,2.5,north,1.5,True,0.0,60.0,True\nproportional,5.0,S4,perpendicular,2.6667,2.5,east,0.75,True,0.0,29.17,False\nproportional,5.0,S4,perpendicular,2.6667,2.5,east,1.0,True,0.0,41.67,False\nproportional,5.0,S4,perpendicular,2.6667,2.5,east,1.5,True,0.0,60.0,True\nproportional,5.0,S4,perpendicular,2.6667,2.5,north,0.75,True,0.0,25.0,False\nproportional,5.0,S4,perpendicular,2.6667,2.5,north,1.0,True,0.0,50.0,False\nproportional,5.0,S4,perpendicular,2.6667,2.5,north,1.5,True,0.0,100.0,True\nproportional,5.0,S5,parallel,3.5833,2.5,east,0.75,True,0.0,0.0,False\nproportional,5.0,S5,parallel,3.5833,2.5,east,1.0,True,0.0,0.0,False\nproportional,5.0,S5,parallel,3.5833,2.5,east,1.5,True,0.0,0.0,False\nproportional,5.0,S5,parallel,3.5833,2.5,north,0.75,True,0.0,37.5,False\nproportional,5.0,S5,parallel,3.5833,2.5,north,1.0,True,0.0,50.0,False\nproportional,5.0,S5,parallel,3.5833,2.5,north,1.5,True,0.0,60.0,True\nproportional,5.0,S6,perpendicular,3.5833,2.5,east,0.75,True,0.0,0.0,False\nproportional,5.0,S6,perpendicular,3.5833,2.5,east,1.0,True,0.0,0.0,False\nproportional,5.0,S6,perpendicular,3.5833,2.5,east,1.5,True,0.0,20.83,False\nproportional,5.0,S6,perpendicular,3.5833,2.5,north,0.75,True,0.0,25.0,False\nproportional,5.0,S6,perpendicular,3.5833,2.5,north,1.0,True,0.0,50.0,False\nproportional,5.0,S6,perpendicular,3.5833,2.5,north,1.5,True,0.0,100.0,True\nproportional,5.5,S1,parallel,2.0167,2.75,east,0.75,True,0.0,73.33,True\nproportional,5.5,S1,parallel,2.0167,2.75,east,1.0,True,0.0,98.33,True\nproportional,5.5,S1,parallel,2.0167,2.75,east,1.5,True,0.0,71.67,True\nproportional,5.5,S1,parallel,2.0167,2.75,north,0.75,True,0.0,25.0,False\nproportional,5.5,S1,parallel,2.0167,2.75,north,1.0,True,0.0,37.5,False\nproportional,5.5,S1,parallel,2.0167,2.75,north,1.5,True,0.0,60.0,True\nproportional,5.5,S2,perpendicular,2.0167,2.75,east,0.75,True,0.0,60.0,True\nproportional,5.5,S2,perpendicular,2.0167,2.75,east,1.0,True,0.0,60.0,True\nproportional,5.5,S2,perpendicular,2.0167,2.75,east,1.5,True,0.4833,60.0,False\nproportional,5.5,S2,perpendicular,2.0167,2.75,north,0.75,True,0.0,0.0,False\nproportional,5.5,S2,perpendicular,2.0167,2.75,north,1.0,True,0.0,25.0,False\nproportional,5.5,S2,perpendicular,2.0167,2.75,north,1.5,True,0.0,75.0,True\nproportional,5.5,S3,parallel,2.9333,2.75,east,0.75,True,0.0,0.0,False\nproportional,5.5,S3,parallel,2.9333,2.75,east,1.0,True,0.0,6.67,False\nproportional,5.5,S3,parallel,2.9333,2.75,east,1.5,True,0.0,56.67,False\nproportional,5.5,S3,parallel,2.9333,2.75,north,0.75,True,0.0,25.0,False\nproportional,5.5,S3,parallel,2.9333,2.75,north,1.0,True,0.0,37.5,False\nproportional,5.5,S3,parallel,2.9333,2.75,north,1.5,True,0.0,60.0,True\nproportional,5.5,S4,perpendicular,2.9333,2.75,east,0.75,True,0.0,15.83,False\nproportional,5.5,S4,perpendicular,2.9333,2.75,east,1.0,True,0.0,28.33,False\nproportional,5.5,S4,perpendicular,2.9333,2.75,east,1.5,True,0.0,53.33,False\nproportional,5.5,S4,perpendicular,2.9333,2.75,north,0.75,True,0.0,0.0,False\nproportional,5.5,S4,perpendicular,2.9333,2.75,north,1.0,True,0.0,25.0,False\nproportional,5.5,S4,perpendicular,2.9333,2.75,north,1.5,True,0.0,75.0,True\nproportional,5.5,S5,parallel,3.9417,2.75,east,0.75,True,0.0,0.0,False\nproportional,5.5,S5,parallel,3.9417,2.75,east,1.0,True,0.0,0.0,False\nproportional,5.5,S5,parallel,3.9417,2.75,east,1.5,True,0.0,0.0,False\nproportional,5.5,S5,parallel,3.9417,2.75,north,0.75,True,0.0,25.0,False\nproportional,5.5,S5,parallel,3.9417,2.75,north,1.0,True,0.0,37.5,False\nproportional,5.5,S5,parallel,3.9417,2.75,north,1.5,True,0.0,60.0,True\nproportional,5.5,S6,perpendicular,3.9417,2.75,east,0.75,True,0.0,0.0,False\nproportional,5.5,S6,perpendicular,3.9417,2.75,east,1.0,True,0.0,0.0,False\nproportional,5.5,S6,perpendicular,3.9417,2.75,east,1.5,True,0.0,2.92,False\nproportional,5.5,S6,perpendicular,3.9417,2.75,north,0.75,True,0.0,0.0,False\nproportional,5.5,S6,perpendicular,3.9417,2.75,north,1.0,True,0.0,25.0,False\nproportional,5.5,S6,perpendicular,3.9417,2.75,north,1.5,True,0.0,75.0,True\nproportional,6.0,S1,parallel,2.2,3.0,east,0.75,True,0.0,55.0,False\nproportional,6.0,S1,parallel,2.2,3.0,east,1.0,True,0.0,80.0,True\nproportional,6.0,S1,parallel,2.2,3.0,east,1.5,True,0.0,90.0,True\nproportional,6.0,S1,parallel,2.2,3.0,north,0.75,True,0.0,12.5,False\nproportional,6.0,S1,parallel,2.2,3.0,north,1.0,True,0.0,25.0,False\nproportional,6.0,S1,parallel,2.2,3.0,north,1.5,True,0.0,50.0,False\nproportional,6.0,S2,perpendicular,2.2,3.0,east,0.75,True,0.0,52.5,False\nproportional,6.0,S2,perpendicular,2.2,3.0,east,1.0,True,0.0,60.0,True\nproportional,6.0,S2,perpendicular,2.2,3.0,east,1.5,True,0.3,60.0,False\nproportional,6.0,S2,perpendicular,2.2,3.0,north,0.75,True,0.0,0.0,False\nproportional,6.0,S2,perpendicular,2.2,3.0,north,1.0,True,0.0,0.0,False\nproportional,6.0,S2,perpendicular,2.2,3.0,north,1.5,True,0.0,50.0,False\nproportional,6.0,S3,parallel,3.2,3.0,east,0.75,True,0.0,0.0,False\nproportional,6.0,S3,parallel,3.2,3.0,east,1.0,True,0.0,0.0,False\nproportional,6.0,S3,parallel,3.2,3.0,east,1.5,True,0.0,30.0,False\nproportional,6.0,S3,parallel,3.2,3.0,north,0.75,True,0.0,12.5,False\nproportional,6.0,S3,parallel,3.2,3.0,north,1.0,True,0.0,25.0,False\nproportional,6.0,S3,parallel,3.2,3.0,north,1.5,True,0.0,50.0,False\nproportional,6.0,S4,perpendicular,3.2,3.0,east,0.75,True,0.0,2.5,False\nproportional,6.0,S4,perpendicular,3.2,3.0,east,1.0,True,0.0,15.0,False\nproportional,6.0,S4,perpendicular,3.2,3.0,east,1.5,True,0.0,40.0,False\nproportional,6.0,S4,perpendicular,3.2,3.0,north,0.75,True,0.0,0.0,False\nproportional,6.0,S4,perpendicular,3.2,3.0,north,1.0,True,0.0,0.0,False\nproportional,6.0,S4,perpendicular,3.2,3.0,north,1.5,True,0.0,50.0,False\nproportional,6.0,S5,parallel,4.3,3.0,east,0.75,True,0.0,0.0,False\nproportional,6.0,S5,parallel,4.3,3.0,east,1.0,True,0.0,0.0,False\nproportional,6.0,S5,parallel,4.3,3.0,east,1.5,True,0.0,0.0,False\nproportional,6.0,S5,parallel,4.3,3.0,north,0.75,True,0.0,12.5,False\nproportional,6.0,S5,parallel,4.3,3.0,north,1.0,True,0.0,25.0,False\nproportional,6.0,S5,parallel,4.3,3.0,north,1.5,True,0.0,50.0,False\nproportional,6.0,S6,perpendicular,4.3,3.0,east,0.75,True,0.0,0.0,False\nproportional,6.0,S6,perpendicular,4.3,3.0,east,1.0,True,0.0,0.0,False\nproportional,6.0,S6,perpendicular,4.3,3.0,east,1.5,True,0.0,0.0,False\nproportional,6.0,S6,perpendicular,4.3,3.0,north,0.75,True,0.0,0.0,False\nproportional,6.0,S6,perpendicular,4.3,3.0,north,1.0,True,0.0,0.0,False\nproportional,6.0,S6,perpendicular,4.3,3.0,north,1.5,True,0.0,50.0,False\nproportional,6.5,S1,parallel,2.3833,3.25,east,0.75,True,0.0,36.67,False\nproportional,6.5,S1,parallel,2.3833,3.25,east,1.0,True,0.0,61.67,True\nproportional,6.5,S1,parallel,2.3833,3.25,east,1.5,True,0.0,100.0,True\nproportional,6.5,S1,parallel,2.3833,3.25,north,0.75,True,0.0,0.0,False\nproportional,6.5,S1,parallel,2.3833,3.25,north,1.0,True,0.0,12.5,False\nproportional,6.5,S1,parallel,2.3833,3.25,north,1.5,True,0.0,37.5,False\nproportional,6.5,S2,perpendicular,2.3833,3.25,east,0.75,True,0.0,43.33,False\nproportional,6.5,S2,perpendicular,2.3833,3.25,east,1.0,True,0.0,55.83,False\nproportional,6.5,S2,perpendicular,2.3833,3.25,east,1.5,True,0.1167,60.0,False\nproportional,6.5,S2,perpendicular,2.3833,3.25,north,0.75,True,0.0,0.0,False\nproportional,6.5,S2,perpendicular,2.3833,3.25,north,1.0,True,0.0,0.0,False\nproportional,6.5,S2,perpendicular,2.3833,3.25,north,1.5,True,0.0,25.0,False\nproportional,6.5,S3,parallel,3.4667,3.25,east,0.75,True,0.0,0.0,False\nproportional,6.5,S3,parallel,3.4667,3.25,east,1.0,True,0.0,0.0,False\nproportional,6.5,S3,parallel,3.4667,3.25,east,1.5,True,0.0,3.33,False\nproportional,6.5,S3,parallel,3.4667,3.25,north,0.75,True,0.0,0.0,False\nproportional,6.5,S3,parallel,3.4667,3.25,north,1.0,True,0.0,12.5,False\nproportional,6.5,S3,parallel,3.4667,3.25,north,1.5,True,0.0,37.5,False\nproportional,6.5,S4,perpendicular,3.4667,3.25,east,0.75,True,0.0,0.0,False\nproportional,6.5,S4,perpendicular,3.4667,3.25,east,1.0,True,0.0,1.67,False\nproportional,6.5,S4,perpendicular,3.4667,3.25,east,1.5,True,0.0,26.67,False\nproportional,6.5,S4,perpendicular,3.4667,3.25,north,0.75,True,0.0,0.0,False\nproportional,6.5,S4,perpendicular,3.4667,3.25,north,1.0,True,0.0,0.0,False\nproportional,6.5,S4,perpendicular,3.4667,3.25,north,1.5,True,0.0,25.0,False\nproportional,6.5,S5,parallel,4.6583,3.25,east,0.75,True,0.0,0.0,False\nproportional,6.5,S5,parallel,4.6583,3.25,east,1.0,True,0.0,0.0,False\nproportional,6.5,S5,parallel,4.6583,3.25,east,1.5,True,0.0,0.0,False\nproportional,6.5,S5,parallel,4.6583,3.25,north,0.75,True,0.0,0.0,False\nproportional,6.5,S5,parallel,4.6583,3.25,north,1.0,True,0.0,12.5,False\nproportional,6.5,S5,parallel,4.6583,3.25,north,1.5,True,0.0,37.5,False\nproportional,6.5,S6,perpendicular,4.6583,3.25,east,0.75,True,0.0,0.0,False\nproportional,6.5,S6,perpendicular,4.6583,3.25,east,1.0,True,0.0,0.0,False\nproportional,6.5,S6,perpendicular,4.6583,3.25,east,1.5,True,0.0,0.0,False\nproportional,6.5,S6,perpendicular,4.6583,3.25,north,0.75,True,0.0,0.0,False\nproportional,6.5,S6,perpendicular,4.6583,3.25,north,1.0,True,0.0,0.0,False\nproportional,6.5,S6,perpendicular,4.6583,3.25,north,1.5,True,0.0,25.0,False\nproportional,7.0,S1,parallel,2.5667,3.5,east,0.75,True,0.0,18.33,False\nproportional,7.0,S1,parallel,2.5667,3.5,east,1.0,True,0.0,43.33,False\nproportional,7.0,S1,parallel,2.5667,3.5,east,1.5,True,0.0,93.33,True\nproportional,7.0,S1,parallel,2.5667,3.5,north,0.75,True,0.0,0.0,False\nproportional,7.0,S1,parallel,2.5667,3.5,north,1.0,True,0.0,0.0,False\nproportional,7.0,S1,parallel,2.5667,3.5,north,1.5,True,0.0,25.0,False\nproportional,7.0,S2,perpendicular,2.5667,3.5,east,0.75,True,0.0,34.17,False\nproportional,7.0,S2,perpendicular,2.5667,3.5,east,1.0,True,0.0,46.67,False\nproportional,7.0,S2,perpendicular,2.5667,3.5,east,1.5,True,0.0,60.0,True\nproportional,7.0,S2,perpendicular,2.5667,3.5,north,0.75,True,0.0,0.0,False\nproportional,7.0,S2,perpendicular,2.5667,3.5,north,1.0,True,0.0,0.0,False\nproportional,7.0,S2,perpendicular,2.5667,3.5,north,1.5,True,0.0,0.0,False\nproportional,7.0,S3,parallel,3.7333,3.5,east,0.75,True,0.0,0.0,False\nproportional,7.0,S3,parallel,3.7333,3.5,east,1.0,True,0.0,0.0,False\nproportional,7.0,S3,parallel,3.7333,3.5,east,1.5,True,0.0,0.0,False\nproportional,7.0,S3,parallel,3.7333,3.5,north,0.75,True,0.0,0.0,False\nproportional,7.0,S3,parallel,3.7333,3.5,north,1.0,True,0.0,0.0,False\nproportional,7.0,S3,parallel,3.7333,3.5,north,1.5,True,0.0,25.0,False\nproportional,7.0,S4,perpendicular,3.7333,3.5,east,0.75,True,0.0,0.0,False\nproportional,7.0,S4,perpendicular,3.7333,3.5,east,1.0,True,0.0,0.0,False\nproportional,7.0,S4,perpendicular,3.7333,3.5,east,1.5,True,0.0,13.33,False\nproportional,7.0,S4,perpendicular,3.7333,3.5,north,0.75,True,0.0,0.0,False\nproportional,7.0,S4,perpendicular,3.7333,3.5,north,1.0,True,0.0,0.0,False\nproportional,7.0,S4,perpendicular,3.7333,3.5,north,1.5,True,0.0,0.0,False\nproportional,7.0,S5,parallel,5.0167,3.5,east,0.75,True,0.0,0.0,False\nproportional,7.0,S5,parallel,5.0167,3.5,east,1.0,True,0.0,0.0,False\nproportional,7.0,S5,parallel,5.0167,3.5,east,1.5,True,0.0,0.0,False\nproportional,7.0,S5,parallel,5.0167,3.5,north,0.75,True,0.0,0.0,False\nproportional,7.0,S5,parallel,5.0167,3.5,north,1.0,True,0.0,0.0,False\nproportional,7.0,S5,parallel,5.0167,3.5,north,1.5,True,0.0,25.0,False\nproportional,7.0,S6,perpendicular,5.0167,3.5,east,0.75,True,0.0,0.0,False\nproportional,7.0,S6,perpendicular,5.0167,3.5,east,1.0,True,0.0,0.0,False\nproportional,7.0,S6,perpendicular,5.0167,3.5,east,1.5,True,0.0,0.0,False\nproportional,7.0,S6,perpendicular,5.0167,3.5,north,0.75,True,0.0,0.0,False\nproportional,7.0,S6,perpendicular,5.0167,3.5,north,1.0,True,0.0,0.0,False\nproportional,7.0,S6,perpendicular,5.0167,3.5,north,1.5,True,0.0,0.0,False\nfixed_offset,4.0,S1,parallel,2.2,3.0,east,0.75,True,0.0,55.0,False\nfixed_offset,4.0,S1,parallel,2.2,3.0,east,1.0,True,0.0,80.0,True\nfixed_offset,4.0,S1,parallel,2.2,3.0,east,1.5,True,0.0,90.0,True\nfixed_offset,4.0,S1,parallel,2.2,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,4.0,S1,parallel,2.2,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,4.0,S1,parallel,2.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,4.0,S2,perpendicular,2.2,3.0,east,0.75,True,0.0,52.5,False\nfixed_offset,4.0,S2,perpendicular,2.2,3.0,east,1.0,True,0.0,60.0,True\nfixed_offset,4.0,S2,perpendicular,2.2,3.0,east,1.5,True,0.3,60.0,False\nfixed_offset,4.0,S2,perpendicular,2.2,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,4.0,S2,perpendicular,2.2,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,4.0,S2,perpendicular,2.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,4.0,S3,parallel,3.2,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,4.0,S3,parallel,3.2,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,4.0,S3,parallel,3.2,3.0,east,1.5,True,0.0,30.0,False\nfixed_offset,4.0,S3,parallel,3.2,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,4.0,S3,parallel,3.2,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,4.0,S3,parallel,3.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,4.0,S4,perpendicular,3.2,3.0,east,0.75,False,,,False\nfixed_offset,4.0,S4,perpendicular,3.2,3.0,east,1.0,False,,,False\nfixed_offset,4.0,S4,perpendicular,3.2,3.0,east,1.5,False,,,False\nfixed_offset,4.0,S4,perpendicular,3.2,3.0,north,0.75,False,,,False\nfixed_offset,4.0,S4,perpendicular,3.2,3.0,north,1.0,False,,,False\nfixed_offset,4.0,S4,perpendicular,3.2,3.0,north,1.5,False,,,False\nfixed_offset,4.0,S5,parallel,4.3,3.0,east,0.75,False,,,False\nfixed_offset,4.0,S5,parallel,4.3,3.0,east,1.0,False,,,False\nfixed_offset,4.0,S5,parallel,4.3,3.0,east,1.5,False,,,False\nfixed_offset,4.0,S5,parallel,4.3,3.0,north,0.75,False,,,False\nfixed_offset,4.0,S5,parallel,4.3,3.0,north,1.0,False,,,False\nfixed_offset,4.0,S5,parallel,4.3,3.0,north,1.5,False,,,False\nfixed_offset,4.0,S6,perpendicular,4.3,3.0,east,0.75,False,,,False\nfixed_offset,4.0,S6,perpendicular,4.3,3.0,east,1.0,False,,,False\nfixed_offset,4.0,S6,perpendicular,4.3,3.0,east,1.5,False,,,False\nfixed_offset,4.0,S6,perpendicular,4.3,3.0,north,0.75,False,,,False\nfixed_offset,4.0,S6,perpendicular,4.3,3.0,north,1.0,False,,,False\nfixed_offset,4.0,S6,perpendicular,4.3,3.0,north,1.5,False,,,False\nfixed_offset,4.5,S1,parallel,2.2,3.0,east,0.75,True,0.0,55.0,False\nfixed_offset,4.5,S1,parallel,2.2,3.0,east,1.0,True,0.0,80.0,True\nfixed_offset,4.5,S1,parallel,2.2,3.0,east,1.5,True,0.0,90.0,True\nfixed_offset,4.5,S1,parallel,2.2,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,4.5,S1,parallel,2.2,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,4.5,S1,parallel,2.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,4.5,S2,perpendicular,2.2,3.0,east,0.75,True,0.0,52.5,False\nfixed_offset,4.5,S2,perpendicular,2.2,3.0,east,1.0,True,0.0,60.0,True\nfixed_offset,4.5,S2,perpendicular,2.2,3.0,east,1.5,True,0.3,60.0,False\nfixed_offset,4.5,S2,perpendicular,2.2,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,4.5,S2,perpendicular,2.2,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,4.5,S2,perpendicular,2.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,4.5,S3,parallel,3.2,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,4.5,S3,parallel,3.2,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,4.5,S3,parallel,3.2,3.0,east,1.5,True,0.0,30.0,False\nfixed_offset,4.5,S3,parallel,3.2,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,4.5,S3,parallel,3.2,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,4.5,S3,parallel,3.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,4.5,S4,perpendicular,3.2,3.0,east,0.75,True,0.0,2.5,False\nfixed_offset,4.5,S4,perpendicular,3.2,3.0,east,1.0,True,0.0,15.0,False\nfixed_offset,4.5,S4,perpendicular,3.2,3.0,east,1.5,True,0.0,40.0,False\nfixed_offset,4.5,S4,perpendicular,3.2,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,4.5,S4,perpendicular,3.2,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,4.5,S4,perpendicular,3.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,4.5,S5,parallel,4.3,3.0,east,0.75,False,,,False\nfixed_offset,4.5,S5,parallel,4.3,3.0,east,1.0,False,,,False\nfixed_offset,4.5,S5,parallel,4.3,3.0,east,1.5,False,,,False\nfixed_offset,4.5,S5,parallel,4.3,3.0,north,0.75,False,,,False\nfixed_offset,4.5,S5,parallel,4.3,3.0,north,1.0,False,,,False\nfixed_offset,4.5,S5,parallel,4.3,3.0,north,1.5,False,,,False\nfixed_offset,4.5,S6,perpendicular,4.3,3.0,east,0.75,False,,,False\nfixed_offset,4.5,S6,perpendicular,4.3,3.0,east,1.0,False,,,False\nfixed_offset,4.5,S6,perpendicular,4.3,3.0,east,1.5,False,,,False\nfixed_offset,4.5,S6,perpendicular,4.3,3.0,north,0.75,False,,,False\nfixed_offset,4.5,S6,perpendicular,4.3,3.0,north,1.0,False,,,False\nfixed_offset,4.5,S6,perpendicular,4.3,3.0,north,1.5,False,,,False\nfixed_offset,5.0,S1,parallel,2.2,3.0,east,0.75,True,0.0,55.0,False\nfixed_offset,5.0,S1,parallel,2.2,3.0,east,1.0,True,0.0,80.0,True\nfixed_offset,5.0,S1,parallel,2.2,3.0,east,1.5,True,0.0,90.0,True\nfixed_offset,5.0,S1,parallel,2.2,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,5.0,S1,parallel,2.2,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,5.0,S1,parallel,2.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,5.0,S2,perpendicular,2.2,3.0,east,0.75,True,0.0,52.5,False\nfixed_offset,5.0,S2,perpendicular,2.2,3.0,east,1.0,True,0.0,60.0,True\nfixed_offset,5.0,S2,perpendicular,2.2,3.0,east,1.5,True,0.3,60.0,False\nfixed_offset,5.0,S2,perpendicular,2.2,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,5.0,S2,perpendicular,2.2,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,5.0,S2,perpendicular,2.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,5.0,S3,parallel,3.2,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,5.0,S3,parallel,3.2,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,5.0,S3,parallel,3.2,3.0,east,1.5,True,0.0,30.0,False\nfixed_offset,5.0,S3,parallel,3.2,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,5.0,S3,parallel,3.2,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,5.0,S3,parallel,3.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,5.0,S4,perpendicular,3.2,3.0,east,0.75,True,0.0,2.5,False\nfixed_offset,5.0,S4,perpendicular,3.2,3.0,east,1.0,True,0.0,15.0,False\nfixed_offset,5.0,S4,perpendicular,3.2,3.0,east,1.5,True,0.0,40.0,False\nfixed_offset,5.0,S4,perpendicular,3.2,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,5.0,S4,perpendicular,3.2,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,5.0,S4,perpendicular,3.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,5.0,S5,parallel,4.3,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,5.0,S5,parallel,4.3,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,5.0,S5,parallel,4.3,3.0,east,1.5,True,0.0,0.0,False\nfixed_offset,5.0,S5,parallel,4.3,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,5.0,S5,parallel,4.3,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,5.0,S5,parallel,4.3,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,5.0,S6,perpendicular,4.3,3.0,east,0.75,False,,,False\nfixed_offset,5.0,S6,perpendicular,4.3,3.0,east,1.0,False,,,False\nfixed_offset,5.0,S6,perpendicular,4.3,3.0,east,1.5,False,,,False\nfixed_offset,5.0,S6,perpendicular,4.3,3.0,north,0.75,False,,,False\nfixed_offset,5.0,S6,perpendicular,4.3,3.0,north,1.0,False,,,False\nfixed_offset,5.0,S6,perpendicular,4.3,3.0,north,1.5,False,,,False\nfixed_offset,5.5,S1,parallel,2.2,3.0,east,0.75,True,0.0,55.0,False\nfixed_offset,5.5,S1,parallel,2.2,3.0,east,1.0,True,0.0,80.0,True\nfixed_offset,5.5,S1,parallel,2.2,3.0,east,1.5,True,0.0,90.0,True\nfixed_offset,5.5,S1,parallel,2.2,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,5.5,S1,parallel,2.2,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,5.5,S1,parallel,2.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,5.5,S2,perpendicular,2.2,3.0,east,0.75,True,0.0,52.5,False\nfixed_offset,5.5,S2,perpendicular,2.2,3.0,east,1.0,True,0.0,60.0,True\nfixed_offset,5.5,S2,perpendicular,2.2,3.0,east,1.5,True,0.3,60.0,False\nfixed_offset,5.5,S2,perpendicular,2.2,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,5.5,S2,perpendicular,2.2,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,5.5,S2,perpendicular,2.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,5.5,S3,parallel,3.2,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,5.5,S3,parallel,3.2,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,5.5,S3,parallel,3.2,3.0,east,1.5,True,0.0,30.0,False\nfixed_offset,5.5,S3,parallel,3.2,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,5.5,S3,parallel,3.2,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,5.5,S3,parallel,3.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,5.5,S4,perpendicular,3.2,3.0,east,0.75,True,0.0,2.5,False\nfixed_offset,5.5,S4,perpendicular,3.2,3.0,east,1.0,True,0.0,15.0,False\nfixed_offset,5.5,S4,perpendicular,3.2,3.0,east,1.5,True,0.0,40.0,False\nfixed_offset,5.5,S4,perpendicular,3.2,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,5.5,S4,perpendicular,3.2,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,5.5,S4,perpendicular,3.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,5.5,S5,parallel,4.3,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,5.5,S5,parallel,4.3,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,5.5,S5,parallel,4.3,3.0,east,1.5,True,0.0,0.0,False\nfixed_offset,5.5,S5,parallel,4.3,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,5.5,S5,parallel,4.3,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,5.5,S5,parallel,4.3,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,5.5,S6,perpendicular,4.3,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,5.5,S6,perpendicular,4.3,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,5.5,S6,perpendicular,4.3,3.0,east,1.5,True,0.0,0.0,False\nfixed_offset,5.5,S6,perpendicular,4.3,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,5.5,S6,perpendicular,4.3,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,5.5,S6,perpendicular,4.3,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,6.0,S1,parallel,2.2,3.0,east,0.75,True,0.0,55.0,False\nfixed_offset,6.0,S1,parallel,2.2,3.0,east,1.0,True,0.0,80.0,True\nfixed_offset,6.0,S1,parallel,2.2,3.0,east,1.5,True,0.0,90.0,True\nfixed_offset,6.0,S1,parallel,2.2,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,6.0,S1,parallel,2.2,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,6.0,S1,parallel,2.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,6.0,S2,perpendicular,2.2,3.0,east,0.75,True,0.0,52.5,False\nfixed_offset,6.0,S2,perpendicular,2.2,3.0,east,1.0,True,0.0,60.0,True\nfixed_offset,6.0,S2,perpendicular,2.2,3.0,east,1.5,True,0.3,60.0,False\nfixed_offset,6.0,S2,perpendicular,2.2,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,6.0,S2,perpendicular,2.2,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,6.0,S2,perpendicular,2.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,6.0,S3,parallel,3.2,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,6.0,S3,parallel,3.2,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,6.0,S3,parallel,3.2,3.0,east,1.5,True,0.0,30.0,False\nfixed_offset,6.0,S3,parallel,3.2,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,6.0,S3,parallel,3.2,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,6.0,S3,parallel,3.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,6.0,S4,perpendicular,3.2,3.0,east,0.75,True,0.0,2.5,False\nfixed_offset,6.0,S4,perpendicular,3.2,3.0,east,1.0,True,0.0,15.0,False\nfixed_offset,6.0,S4,perpendicular,3.2,3.0,east,1.5,True,0.0,40.0,False\nfixed_offset,6.0,S4,perpendicular,3.2,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,6.0,S4,perpendicular,3.2,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,6.0,S4,perpendicular,3.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,6.0,S5,parallel,4.3,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,6.0,S5,parallel,4.3,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,6.0,S5,parallel,4.3,3.0,east,1.5,True,0.0,0.0,False\nfixed_offset,6.0,S5,parallel,4.3,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,6.0,S5,parallel,4.3,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,6.0,S5,parallel,4.3,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,6.0,S6,perpendicular,4.3,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,6.0,S6,perpendicular,4.3,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,6.0,S6,perpendicular,4.3,3.0,east,1.5,True,0.0,0.0,False\nfixed_offset,6.0,S6,perpendicular,4.3,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,6.0,S6,perpendicular,4.3,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,6.0,S6,perpendicular,4.3,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,6.5,S1,parallel,2.2,3.0,east,0.75,True,0.0,55.0,False\nfixed_offset,6.5,S1,parallel,2.2,3.0,east,1.0,True,0.0,80.0,True\nfixed_offset,6.5,S1,parallel,2.2,3.0,east,1.5,True,0.0,90.0,True\nfixed_offset,6.5,S1,parallel,2.2,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,6.5,S1,parallel,2.2,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,6.5,S1,parallel,2.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,6.5,S2,perpendicular,2.2,3.0,east,0.75,True,0.0,52.5,False\nfixed_offset,6.5,S2,perpendicular,2.2,3.0,east,1.0,True,0.0,60.0,True\nfixed_offset,6.5,S2,perpendicular,2.2,3.0,east,1.5,True,0.3,60.0,False\nfixed_offset,6.5,S2,perpendicular,2.2,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,6.5,S2,perpendicular,2.2,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,6.5,S2,perpendicular,2.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,6.5,S3,parallel,3.2,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,6.5,S3,parallel,3.2,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,6.5,S3,parallel,3.2,3.0,east,1.5,True,0.0,30.0,False\nfixed_offset,6.5,S3,parallel,3.2,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,6.5,S3,parallel,3.2,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,6.5,S3,parallel,3.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,6.5,S4,perpendicular,3.2,3.0,east,0.75,True,0.0,2.5,False\nfixed_offset,6.5,S4,perpendicular,3.2,3.0,east,1.0,True,0.0,15.0,False\nfixed_offset,6.5,S4,perpendicular,3.2,3.0,east,1.5,True,0.0,40.0,False\nfixed_offset,6.5,S4,perpendicular,3.2,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,6.5,S4,perpendicular,3.2,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,6.5,S4,perpendicular,3.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,6.5,S5,parallel,4.3,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,6.5,S5,parallel,4.3,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,6.5,S5,parallel,4.3,3.0,east,1.5,True,0.0,0.0,False\nfixed_offset,6.5,S5,parallel,4.3,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,6.5,S5,parallel,4.3,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,6.5,S5,parallel,4.3,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,6.5,S6,perpendicular,4.3,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,6.5,S6,perpendicular,4.3,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,6.5,S6,perpendicular,4.3,3.0,east,1.5,True,0.0,0.0,False\nfixed_offset,6.5,S6,perpendicular,4.3,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,6.5,S6,perpendicular,4.3,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,6.5,S6,perpendicular,4.3,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,7.0,S1,parallel,2.2,3.0,east,0.75,True,0.0,55.0,False\nfixed_offset,7.0,S1,parallel,2.2,3.0,east,1.0,True,0.0,80.0,True\nfixed_offset,7.0,S1,parallel,2.2,3.0,east,1.5,True,0.0,90.0,True\nfixed_offset,7.0,S1,parallel,2.2,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,7.0,S1,parallel,2.2,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,7.0,S1,parallel,2.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,7.0,S2,perpendicular,2.2,3.0,east,0.75,True,0.0,52.5,False\nfixed_offset,7.0,S2,perpendicular,2.2,3.0,east,1.0,True,0.0,60.0,True\nfixed_offset,7.0,S2,perpendicular,2.2,3.0,east,1.5,True,0.3,60.0,False\nfixed_offset,7.0,S2,perpendicular,2.2,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,7.0,S2,perpendicular,2.2,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,7.0,S2,perpendicular,2.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,7.0,S3,parallel,3.2,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,7.0,S3,parallel,3.2,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,7.0,S3,parallel,3.2,3.0,east,1.5,True,0.0,30.0,False\nfixed_offset,7.0,S3,parallel,3.2,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,7.0,S3,parallel,3.2,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,7.0,S3,parallel,3.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,7.0,S4,perpendicular,3.2,3.0,east,0.75,True,0.0,2.5,False\nfixed_offset,7.0,S4,perpendicular,3.2,3.0,east,1.0,True,0.0,15.0,False\nfixed_offset,7.0,S4,perpendicular,3.2,3.0,east,1.5,True,0.0,40.0,False\nfixed_offset,7.0,S4,perpendicular,3.2,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,7.0,S4,perpendicular,3.2,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,7.0,S4,perpendicular,3.2,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,7.0,S5,parallel,4.3,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,7.0,S5,parallel,4.3,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,7.0,S5,parallel,4.3,3.0,east,1.5,True,0.0,0.0,False\nfixed_offset,7.0,S5,parallel,4.3,3.0,north,0.75,True,0.0,12.5,False\nfixed_offset,7.0,S5,parallel,4.3,3.0,north,1.0,True,0.0,25.0,False\nfixed_offset,7.0,S5,parallel,4.3,3.0,north,1.5,True,0.0,50.0,False\nfixed_offset,7.0,S6,perpendicular,4.3,3.0,east,0.75,True,0.0,0.0,False\nfixed_offset,7.0,S6,perpendicular,4.3,3.0,east,1.0,True,0.0,0.0,False\nfixed_offset,7.0,S6,perpendicular,4.3,3.0,east,1.5,True,0.0,0.0,False\nfixed_offset,7.0,S6,perpendicular,4.3,3.0,north,0.75,True,0.0,0.0,False\nfixed_offset,7.0,S6,perpendicular,4.3,3.0,north,1.0,True,0.0,0.0,False\nfixed_offset,7.0,S6,perpendicular,4.3,3.0,north,1.5,True,0.0,50.0,False\n"}
here = Path.cwd()
if (here / 'run_all.py').exists() and (here / 'a3src' / 'a4_window_size.py').exists():
    ROOT = here
    print('Using repository files in', ROOT)
else:
    ROOT = here / 'A4_rerun'
    for rel, text in FILES.items():
        path = ROOT / rel
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding='utf8')
    print('Wrote embedded files to', ROOT)
print((ROOT / 'inputs' / 'config.json').read_text(encoding='utf8'))

## 3. Run everything
One command: scenarios, window-size test, 39 checks, figures, reproduction check, run record.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / 'run_all.py')], cwd=str(ROOT), capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-2000:])
assert r.returncode == 0, 'A check failed or the results differ from the reference - read the lines above.'

## 4. Key figure (Object Card)
Re-drawn from `outputs/a4_window_size_summary.csv` and `outputs/a4_offset_rule.csv` by the run above.

In [ ]:
from IPython.display import display, Image
display(Image(filename=str(ROOT / 'outputs' / 'fig_window_size.png'), width=980))
display(Image(filename=str(ROOT / 'outputs' / 'fig_window_plans.png'), width=980))

## 5. A3 baseline figures

In [ ]:
for name in ['fig_six_placements.png', 'fig_base_comparison_east.png', 'fig_base_comparison_north.png', 'fig_width_sensitivity.png']:
    display(Image(filename=str(ROOT / 'outputs' / name), width=860))

## 6. Check log
Expected values were written by hand before the code was run.

In [ ]:
import csv
for row in csv.DictReader(open(ROOT / 'outputs' / 'a3_checks.csv', encoding='utf8')):
    print(f"{row['status']:4}  {row['n']:>3}  {row['check']}  |  expected: {row['expected']}  |  observed: {row['observed']}")

## 7. Change an assumption
Edit `inputs/config.json` or `inputs/variants.csv` in the folder printed in section 2, then run section 3 again. The hand-calculated checks and the reference comparison describe the default inputs, so they will report differences; that is expected and shows what changed.